# B2 · Tokenisation et embeddings — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b2_tokenisation/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`. Le notebook télécharge du Hugging Face Hub, s'ils ne sont pas déjà là, les fichiers de six tokenizers (quelques Mo, pas les poids des modèles), puis le modèle all-MiniLM-L6-v2 (≈ 90 Mo) pour B2.28 et B2.29 ; sans connexion, leurs cellules affichent ⏳ et le reste continue. Tout tourne sur CPU (≈ 5 min pour les solutions en mode rapide).

> Travaille dans **ta copie** (`mon_travail/b2_tokenisation/03_notebook.ipynb`, créée par `python tools/start_chapter.py B2`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B2.Q1 à B2.Q12, B2.R1 à B2.R3, B2.1 à B2.6, B2.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | B2.13 | Tokenizers réels : GPT-2, BERT et un tokenizer multilingue sur Holmes et Verne | 📦 | ★★ | 20 |
|  | B2.14 | Prédire : quel texte coûte le plus de tokens, Holmes, Verne, des nombres ou des emoji ? | 🔮 | ★ | 10 |
| B | B2.15 | get_pair_counts et merge_pair | 🔨 | ★★ | 20 |
|  | B2.16 | BPETokenizer.fit : apprendre les fusions, au niveau caractère puis octet | 🔨 | ★★★ | 50 |
|  | B2.17 | encode et decode : fusions appliquées par rang, aller-retour sans perte | 🔨 | ★★★ | 40 |
|  | B2.18 | Le tokenizer qui abîme les accents : trois bugs d'Unicode et de fusions | 🐛 | ★★ | 25 |
|  | B2.19 | Ton BPE contre la bibliothèque tokenizers : mêmes fusions, autre vitesse | 📦 | ★★ | 25 |
| C | B2.20 | Taille du vocabulaire : compression et entropie par token, de 300 à 8 000 tokens | 🔬 | ★★★ | 40 |
|  | B2.21 | Tokenizer entraîné sur Holmes, appliqué à Verne (et l'inverse) : le coût de la langue | 🔬 | ★★ | 25 |
| D | B2.22 | cooccurrence_matrix et ppmi | 🔨 | ★★ | 30 |
|  | B2.23 | cosine_similarity_matrix et most_similar | 🔨 | ★★ | 20 |
|  | B2.24 | Embeddings PPMI + SVD sur Holmes : les voisins de « Holmes », « Watson » et « letter » | 📦 | ★★ | 30 |
|  | B2.25 | word2vec (skip-gram, échantillonnage négatif) en PyTorch sur Verne | 📦 | ★★★ | 60 |
|  | B2.26 | Prédire les voisins de « Fogg » et de « Passepartout » avant de les calculer | 🔮 | ★ | 10 |
|  | B2.27 | Lire une carte PCA/UMAP d'embeddings : groupes, axes et effet de la fréquence | 📈 | ★★ | 20 |
| E | B2.28 | Embeddings de phrases (MiniLM, mean pooling) et recherche sémantique dans Holmes | 📦 | ★★ | 30 |
|  | B2.29 | Statique contre contextuel : le même mot dans deux phrases | 🔬 | ★★ | 25 |
|  | B2.30 | Sauvegarder un tokenizer en JSON et écrire son test de non-régression | 🛠️ | ★★ | 20 |
|  | B2.31 | Défi : la meilleure compression sans perte de Verne avec 2 000 tokens, en moins de 2 min de CPU | 🏆 | ★★★★ | 90 |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")  # your own library

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True
ℹ️ Ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2.


## Objectifs et rappel express

- Manipuler Unicode, UTF-8 et la normalisation d'un texte, et compter ses octets et ses points de code.
- Écrire un tokenizer BPE (entraînement, encodage par rang, décodage) et le comparer à la bibliothèque `tokenizers`.
- Comparer des tokenizers réels et mesurer leur coût (fertilité, compression) selon la langue.
- Relier `nn.Embedding` à un produit one-hot × table, et entraîner des embeddings word2vec.
- Construire des embeddings par co-occurrences (PPMI et SVD), les interroger par le cosinus, et chercher des passages par des embeddings de phrases.

**Rappel express.** Fertilité $= \frac{\text{tokens}}{\text{mots}}$, compression $= \frac{\text{octets}}{\text{tokens}}$. Fusion BPE : la paire voisine la plus fréquente (à égalité, la plus petite paire d'identifiants) devient un token ; on encode en appliquant les fusions par rang. $\mathbf{x}_{\text{one-hot}}^\top \mathbf{E} = \mathbf{E}[i, :]$. $\operatorname{PPMI}(w, c) = \max\big(0, \log \frac{p(w, c)}{p(w)\,p(c)}\big)$. Skip-gram : $L = -\log \sigma(\mathbf{u}_c^\top \mathbf{v}_w) - \sum_k \log \sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_w)$. Mean pooling : $\mathbf{s} = \frac{\sum_t m_t \mathbf{h}_t}{\sum_t m_t}$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B2.1 à B2.6, 🧮 B2.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; une forme de tenseur : entre guillemets (`"(2, 3)"`) ; un mot : entre guillemets (`"mot"`) ; des octets : une liste, en hexadécimal ou en décimal (`[0x4E, 0x6F]` ou `[78, 111]`) ; des identifiants, une paire d'identifiants ou un vecteur : une liste (`[4, 0, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B2.7, l'estimation 🧮 B2.8, la réflexion (🗣️ B2.10, ⚖️ B2.11, 📄 B2.12) et l'entretien se corrigent avec `05_solutions.md`.

**B2.Q1 — Caractères, mots ou sous-mots : taille du vocabulaire contre longueur des séquences**

In [2]:
# B2.Q1: the values you found on paper
answer_B2_Q1a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q1b = ...  # b) True or False
answer_B2_Q1c = ...  # c) a whole number
answer_B2_Q1d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q1a, answer_B2_Q1b, answer_B2_Q1c, answer_B2_Q1d, answer_B2_Q1e]):
    wb.check(f"B2.Q1{letter}", answer)

⏳ Ex B2.Q1a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q1b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q1c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q1d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q1e : pas encore fait (remplace `...` par ta réponse).


**B2.Q2 — Octets, points de code, caractères affichés : combien « é » compte-t-il ?**

In [3]:
# B2.Q2: the values you found on paper
answer_B2_Q2a = ...  # a) a whole number
answer_B2_Q2b = ...  # b) a whole number
answer_B2_Q2c = ...  # c) a whole number
answer_B2_Q2d = ...  # d) a whole number
answer_B2_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q2a, answer_B2_Q2b, answer_B2_Q2c, answer_B2_Q2d, answer_B2_Q2e]):
    wb.check(f"B2.Q2{letter}", answer)

⏳ Ex B2.Q2a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q2b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q2c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q2d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q2e : pas encore fait (remplace `...` par ta réponse).


**B2.Q3 — NFC ou NFD : deux chaînes identiques à l'écran, différentes en mémoire**

In [4]:
# B2.Q3: the values you found on paper
answer_B2_Q3a = ...  # a) True or False
answer_B2_Q3b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q3c = ...  # c) True or False
answer_B2_Q3d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q3a, answer_B2_Q3b, answer_B2_Q3c, answer_B2_Q3d, answer_B2_Q3e]):
    wb.check(f"B2.Q3{letter}", answer)

⏳ Ex B2.Q3a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q3b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q3c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q3d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q3e : pas encore fait (remplace `...` par ta réponse).


**B2.Q4 — Fusions BPE : pourquoi la liste ordonnée des fusions est le modèle**

In [5]:
# B2.Q4: the values you found on paper
answer_B2_Q4a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q4b = ...  # b) True or False
answer_B2_Q4c = ...  # c) a whole number
answer_B2_Q4d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q4a, answer_B2_Q4b, answer_B2_Q4c, answer_B2_Q4d, answer_B2_Q4e]):
    wb.check(f"B2.Q4{letter}", answer)

⏳ Ex B2.Q4a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q4b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q4c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q4d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q4e : pas encore fait (remplace `...` par ta réponse).


**B2.Q5 — BPE au niveau octet : jamais de token inconnu, vrai ou faux ?**

In [6]:
# B2.Q5: the values you found on paper
answer_B2_Q5a = ...  # a) True or False
answer_B2_Q5b = ...  # b) a whole number
answer_B2_Q5c = ...  # c) True or False
answer_B2_Q5d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q5a, answer_B2_Q5b, answer_B2_Q5c, answer_B2_Q5d, answer_B2_Q5e]):
    wb.check(f"B2.Q5{letter}", answer)

⏳ Ex B2.Q5a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q5b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q5c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q5d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q5e : pas encore fait (remplace `...` par ta réponse).


**B2.Q6 — Tokens spéciaux et espace initial : « Ġthe » n'est pas « the »**

In [7]:
# B2.Q6: the values you found on paper
answer_B2_Q6a = ...  # a) True or False
answer_B2_Q6b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q6c = ...  # c) the letter of your choice, e.g. "B"
answer_B2_Q6d = ...  # d) True or False
answer_B2_Q6e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B2_Q6a, answer_B2_Q6b, answer_B2_Q6c, answer_B2_Q6d, answer_B2_Q6e]):
    wb.check(f"B2.Q6{letter}", answer)

⏳ Ex B2.Q6a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q6b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q6c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q6d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q6e : pas encore fait (remplace `...` par ta réponse).


**B2.Q7 — WordPiece, Unigram, SentencePiece : quel critère pour quelle méthode ?**

In [8]:
# B2.Q7: the values you found on paper
answer_B2_Q7a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q7b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q7c = ...  # c) the letter of your choice, e.g. "B"
answer_B2_Q7d = ...  # d) True or False
answer_B2_Q7e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B2_Q7a, answer_B2_Q7b, answer_B2_Q7c, answer_B2_Q7d, answer_B2_Q7e]):
    wb.check(f"B2.Q7{letter}", answer)

⏳ Ex B2.Q7a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q7b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q7c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q7d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q7e : pas encore fait (remplace `...` par ta réponse).


**B2.Q8 — Pourquoi un LLM compte mal les lettres d'un mot et calcule mal avec les chiffres**

In [9]:
# B2.Q8: the values you found on paper
answer_B2_Q8a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q8b = ...  # b) True or False
answer_B2_Q8c = ...  # c) a whole number
answer_B2_Q8d = ...  # d) a whole number
answer_B2_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q8a, answer_B2_Q8b, answer_B2_Q8c, answer_B2_Q8d, answer_B2_Q8e]):
    wb.check(f"B2.Q8{letter}", answer)

⏳ Ex B2.Q8a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q8b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q8c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q8d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q8e : pas encore fait (remplace `...` par ta réponse).


**B2.Q9 — nn.Embedding : une table de consultation entraînable, pas une couche magique**

In [10]:
# B2.Q9: the values you found on paper
answer_B2_Q9a = ...  # a) a shape as a string, e.g. "(2, 3)"
answer_B2_Q9b = ...  # b) a whole number
answer_B2_Q9c = ...  # c) True or False
answer_B2_Q9d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q9a, answer_B2_Q9b, answer_B2_Q9c, answer_B2_Q9d, answer_B2_Q9e]):
    wb.check(f"B2.Q9{letter}", answer)

⏳ Ex B2.Q9a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q9b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q9c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q9d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q9e : pas encore fait (remplace `...` par ta réponse).


**B2.Q10 — L'hypothèse distributionnelle : un mot se définit par ses voisins**

In [11]:
# B2.Q10: the values you found on paper
answer_B2_Q10a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q10b = ...  # b) True or False
answer_B2_Q10c = ...  # c) a number, 2 decimals
answer_B2_Q10d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q10a, answer_B2_Q10b, answer_B2_Q10c, answer_B2_Q10d, answer_B2_Q10e]):
    wb.check(f"B2.Q10{letter}", answer)

⏳ Ex B2.Q10a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q10b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q10c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q10d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q10e : pas encore fait (remplace `...` par ta réponse).


**B2.Q11 — Statique ou contextuel : l'« avocat » du tribunal et celui du marché**

In [12]:
# B2.Q11: the values you found on paper
answer_B2_Q11a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q11b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q11c = ...  # c) True or False
answer_B2_Q11d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q11e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B2_Q11a, answer_B2_Q11b, answer_B2_Q11c, answer_B2_Q11d, answer_B2_Q11e]):
    wb.check(f"B2.Q11{letter}", answer)

⏳ Ex B2.Q11a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q11b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q11c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q11d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q11e : pas encore fait (remplace `...` par ta réponse).


**B2.Q12 — Cosinus ou distance euclidienne pour des embeddings normalisés ?**

In [13]:
# B2.Q12: the values you found on paper
answer_B2_Q12a = ...  # a) a number, 1 decimal
answer_B2_Q12b = ...  # b) True or False
answer_B2_Q12c = ...  # c) a number, 3 decimals
answer_B2_Q12d = ...  # d) True or False
answer_B2_Q12e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B2_Q12a, answer_B2_Q12b, answer_B2_Q12c, answer_B2_Q12d, answer_B2_Q12e]):
    wb.check(f"B2.Q12{letter}", answer)

⏳ Ex B2.Q12a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q12b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q12c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q12d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.Q12e : pas encore fait (remplace `...` par ta réponse).


**B2.R1 — À quoi sert l'embedding d'image extrait d'un réseau pré-entraîné ?**

In [14]:
# B2.R1: the values you found on paper
answer_B2_R1a = ...  # a) True or False
answer_B2_R1b = ...  # b) a whole number
answer_B2_R1c = ...  # c) the letter of your choice, e.g. "B"
answer_B2_R1d = ...  # d) True or False
answer_B2_R1e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B2_R1a, answer_B2_R1b, answer_B2_R1c, answer_B2_R1d, answer_B2_R1e]):
    wb.check(f"B2.R1{letter}", answer)

⏳ Ex B2.R1a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R1b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R1c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R1d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R1e : pas encore fait (remplace `...` par ta réponse).


**B2.R2 — build_vocab, encode, decode et le token inconnu**

In [15]:
# B2.R2: the values you found on paper
answer_B2_R2a = ...  # a) a whole number
answer_B2_R2b = ...  # b) a list of token ids, e.g. [4, 0, 7]
answer_B2_R2c = ...  # c) True or False
answer_B2_R2d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_R2e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B2_R2a, answer_B2_R2b, answer_B2_R2c, answer_B2_R2d, answer_B2_R2e]):
    wb.check(f"B2.R2{letter}", answer)

⏳ Ex B2.R2a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R2b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R2c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R2d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R2e : pas encore fait (remplace `...` par ta réponse).


**B2.R3 — Entropie, Huffman et ce que l'on compresse vraiment**

In [16]:
# B2.R3: the values you found on paper
answer_B2_R3a = ...  # a) a number, 2 decimals
answer_B2_R3b = ...  # b) a whole number
answer_B2_R3c = ...  # c) True or False
answer_B2_R3d = ...  # d) True or False
answer_B2_R3e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B2_R3a, answer_B2_R3b, answer_B2_R3c, answer_B2_R3d, answer_B2_R3e]):
    wb.check(f"B2.R3{letter}", answer)

⏳ Ex B2.R3a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R3b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R3c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R3d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.R3e : pas encore fait (remplace `...` par ta réponse).


**Ex B2.1 — « Noël à Paris » en UTF-8 : octets, points de code, longueur**

In [17]:
# Ex B2.1: the values you found on paper
answer_B2_1a = ...  # a) a whole number
answer_B2_1b = ...  # b) a whole number
answer_B2_1c = ...  # c) a list of byte values, e.g. [0x4E, 0x6F]
answer_B2_1d = ...  # d) a whole number
answer_B2_1e = ...  # e) a whole number
answer_B2_1f = ...  # f) a list of byte values, e.g. [0x4E, 0x6F]
answer_B2_1g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_1a, answer_B2_1b, answer_B2_1c, answer_B2_1d, answer_B2_1e, answer_B2_1f, answer_B2_1g]):
    wb.check(f"B2.1{letter}", answer)

⏳ Ex B2.1a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.1b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.1c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.1d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.1e : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.1f : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.1g : pas encore fait (remplace `...` par ta réponse).


**Ex B2.2 — Analogie à la main : roi − homme + femme sur des vecteurs de dimension 3**

In [18]:
# Ex B2.2: the values you found on paper
answer_B2_2a = ...  # a) a number, 2 decimals
answer_B2_2b = ...  # b) a number, 3 decimals
answer_B2_2c = ...  # c) a number, 3 decimals
answer_B2_2d = ...  # d) a word as a string, e.g. "mot"
answer_B2_2e = ...  # e) a word as a string, e.g. "mot"
answer_B2_2f = ...  # f) a word as a string, e.g. "mot"
answer_B2_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_2a, answer_B2_2b, answer_B2_2c, answer_B2_2d, answer_B2_2e, answer_B2_2f, answer_B2_2g]):
    wb.check(f"B2.2{letter}", answer)

⏳ Ex B2.2a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.2b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.2c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.2d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.2e : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.2f : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.2g : pas encore fait (remplace `...` par ta réponse).


**Ex B2.3 — Trois fusions BPE à la main sur un mini-corpus**

In [19]:
# Ex B2.3: the values you found on paper
answer_B2_3a = ...  # a) a whole number
answer_B2_3b = ...  # b) a whole number
answer_B2_3c = ...  # c) a pair of ids as a list or a tuple, e.g. [2, 7]
answer_B2_3d = ...  # d) a whole number
answer_B2_3e = ...  # e) a pair of ids as a list or a tuple, e.g. [2, 7]
answer_B2_3f = ...  # f) a pair of ids as a list or a tuple, e.g. [2, 7]
answer_B2_3g = ...  # g) a whole number
answer_B2_3h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B2_3a, answer_B2_3b, answer_B2_3c, answer_B2_3d, answer_B2_3e, answer_B2_3f, answer_B2_3g, answer_B2_3h]):
    wb.check(f"B2.3{letter}", answer)

⏳ Ex B2.3a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.3b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.3c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.3d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.3e : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.3f : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.3g : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.3h : pas encore fait (remplace `...` par ta réponse).


**Ex B2.4 — Encoder un mot jamais vu avec des fusions déjà apprises**

In [20]:
# Ex B2.4: the values you found on paper
answer_B2_4a = ...  # a) a list of token ids, e.g. [4, 0, 7]
answer_B2_4b = ...  # b) a list of token ids, e.g. [4, 0, 7]
answer_B2_4c = ...  # c) a list of token ids, e.g. [4, 0, 7]
answer_B2_4d = ...  # d) a list of token ids, e.g. [4, 0, 7]
answer_B2_4e = ...  # e) a list of token ids, e.g. [4, 0, 7]
answer_B2_4f = ...  # f) a whole number
answer_B2_4g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_4a, answer_B2_4b, answer_B2_4c, answer_B2_4d, answer_B2_4e, answer_B2_4f, answer_B2_4g]):
    wb.check(f"B2.4{letter}", answer)

⏳ Ex B2.4a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.4b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.4c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.4d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.4e : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.4f : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.4g : pas encore fait (remplace `...` par ta réponse).


**Ex B2.5 — One-hot fois matrice : nn.Embedding à la main, et quelles lignes reçoivent un gradient**

In [21]:
# Ex B2.5: the values you found on paper
answer_B2_5a = ...  # a) a shape as a string, e.g. "(2, 3)"
answer_B2_5b = ...  # b) a list of numbers, e.g. [0.5, -1, 3]
answer_B2_5c = ...  # c) a number, 1 decimal
answer_B2_5d = ...  # d) a list of numbers, e.g. [0.5, -1, 3]
answer_B2_5e = ...  # e) a whole number
answer_B2_5f = ...  # f) a list of numbers, e.g. [0.5, -1, 3]
answer_B2_5g = ...  # g) True or False
answer_B2_5h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B2_5a, answer_B2_5b, answer_B2_5c, answer_B2_5d, answer_B2_5e, answer_B2_5f, answer_B2_5g, answer_B2_5h]):
    wb.check(f"B2.5{letter}", answer)

⏳ Ex B2.5a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.5b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.5c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.5d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.5e : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.5f : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.5g : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.5h : pas encore fait (remplace `...` par ta réponse).


**Ex B2.6 — Co-occurrences et PPMI à la main (fenêtre de 1, quatre mots)**

In [22]:
# Ex B2.6: the values you found on paper
answer_B2_6a = ...  # a) a whole number
answer_B2_6b = ...  # b) a whole number
answer_B2_6c = ...  # c) a number, 3 decimals
answer_B2_6d = ...  # d) a number, 3 decimals
answer_B2_6e = ...  # e) a number, 3 decimals
answer_B2_6f = ...  # f) a number, 3 decimals
answer_B2_6g = ...  # g) True or False
answer_B2_6h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B2_6a, answer_B2_6b, answer_B2_6c, answer_B2_6d, answer_B2_6e, answer_B2_6f, answer_B2_6g, answer_B2_6h]):
    wb.check(f"B2.6{letter}", answer)

⏳ Ex B2.6a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.6b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.6c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.6d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.6e : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.6f : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.6g : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.6h : pas encore fait (remplace `...` par ta réponse).


**Ex B2.9 — Fermi : poids de la table d'embedding de GPT-2 et d'un LLM à 128 000 tokens**

In [23]:
# Ex B2.9: the values you found on paper
answer_B2_9a = ...  # a) a whole number
answer_B2_9b = ...  # b) a number, 1 decimal
answer_B2_9c = ...  # c) a number, 1 decimal
answer_B2_9d = ...  # d) a whole number
answer_B2_9e = ...  # e) a number, 1 decimal
answer_B2_9f = ...  # f) a whole number
answer_B2_9g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_9a, answer_B2_9b, answer_B2_9c, answer_B2_9d, answer_B2_9e, answer_B2_9f, answer_B2_9g]):
    wb.check(f"B2.9{letter}", answer)

⏳ Ex B2.9a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.9b : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.9c : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.9d : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.9e : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.9f : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.9g : pas encore fait (remplace `...` par ta réponse).


## Partie A · Des tokenizers réels : ce que coûte un texte

Avant d'écrire ton propre BPE, tu prends en main trois tokenizers de modèles réels et tu mesures ce que leur coûtent Holmes et Verne : leur nombre de tokens, leur fertilité (tokens par mot) et leur compression (octets par token, fiche §202.6). La cellule suivante charge les outils de la phase « tokenisation » (parties A à C) :
- `EXCERPT_H` et `EXCERPT_V` : les 50 000 caractères `[100000:150000]` de chaque roman, figés ;
- `HOLMES_TRAIN`, `HOLMES_VAL`, `VERNE_TRAIN`, `VERNE_VAL` : chaque roman coupé à 90 % / 10 %, au début d'une ligne et sans mélange (un tokenizer appris sur l'entraînement ne voit jamais la validation) ;
- `TOKENIZER_REPOS` : le dépôt du Hugging Face Hub et la révision (un commit fixé) de six tokenizers (`data/cards/pretrained.md`) ; seuls leurs fichiers de tokenizer se téléchargent, quelques Mo ;
- `n_bytes`, `reference_char_bpe` (le BPE de la bibliothèque `tokenizers`, pour les vérifications), `truncated_bpe` (partie C).

**Données** : *The Adventures of Sherlock Holmes* (A. Conan Doyle, Project Gutenberg n° 1661, `data/cards/holmes.md`) et *Le Tour du monde en quatre-vingts jours* (J. Verne, n° 800, `data/cards/verne.md`), dans le domaine public ; tu les as déjà découpés en caractères et en mots au ch. 22. Sans connexion au Hub, les cellules qui ont besoin d'un tokenizer réel affichent ⏳ « à valider sur Colab » et le reste du notebook continue.

In [24]:
# Tools for the notebook exercises of the tokenization phase (parts A to C)
import json
import math
import os
import re
import time
import unicodedata
import warnings
from collections import Counter
from types import MappingProxyType

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wb import text as wbtext
from wb.nbtools import StopCheck, checked, filled, fr, refuses, returned, run_pytest_group, same, verdict, written

warnings.filterwarnings("ignore", message=".*IProgress not found.*")    # tqdm without ipywidgets (transformers)
try:
    from transformers.utils import logging as _hf_logging
    _hf_logging.set_verbosity_error()     # no « Token indices sequence length is longer… »: we count, no model reads
except ImportError:                       # transformers absent: the cells that need it say so
    pass

TEST_TOK = "tests/test_chb2_tokenizer.py"     # the tests of your library tokenizer.py
EXCERPT_H = wbtext.excerpt("holmes")           # 50 000 characters of Holmes, [100000:150000] (frozen)
EXCERPT_V = wbtext.excerpt("verne")            # 50 000 characters of Verne, [100000:150000]
HOLMES_TRAIN, HOLMES_VAL = wbtext.train_val("holmes")   # 90 % / 10 %, cut at a line, never shuffled
VERNE_TRAIN, VERNE_VAL = wbtext.train_val("verne")
TOKENIZER_REPOS = {name: (spec["repo"], spec["revision"]) for name, spec in wbtext.TOKENIZERS.items()}


def n_bytes(text):
    """The number of UTF-8 bytes of a text."""
    return len(text.encode("utf-8"))


def need(ex_id, *names):
    """Stop a check cell with ⏳ while a function of an earlier exercise is not defined yet."""
    for name in names:
        if globals().get(name) is None:
            print(f"⏳ Ex {ex_id} : définis d'abord {name}.")
            raise StopCheck


def bpe_written():
    """True once BPETokenizer.fit and encode of your library run (B2.16 and B2.17), else False."""
    return written(lambda: mylearn.tokenizer.BPETokenizer(vocab_size=258, level="byte").fit("aa aa").encode("aa"))


def reference_char_bpe(text, vocab_size, min_frequency=2):
    """The BPE of the library tokenizers trained like BPETokenizer(level="char", pre_tokenizer="whitespace") (fiche
    §202.3): models.BPE, pre_tokenizers.Whitespace, BpeTrainer. For the checks."""
    from tokenizers import Tokenizer, models, pre_tokenizers, trainers
    hf = Tokenizer(models.BPE())
    hf.pre_tokenizer = pre_tokenizers.Whitespace()
    hf.train_from_iterator([text], trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=min_frequency,
                                                       show_progress=False))
    return hf


def fast_byte_bpe(texts, vocab_size):
    """Your BPETokenizer(vocab_size, pre_tokenizer="gpt2"), whose merges are learned by the library tokenizers (Rust):
    the same merges as your fit (wbtext.byte_bpe_merges, B2.19), 50 to 200 times faster than Python. Your encode,
    decode and tokenize do the rest."""
    tok = mylearn.tokenizer.BPETokenizer(vocab_size=vocab_size, pre_tokenizer="gpt2")
    tok.n_base_, tok.merges_, tok.special_ids_ = 256, wbtext.byte_bpe_merges(texts, vocab_size), {}
    tok.vocab_ = {i: bytes([i]) for i in range(256)}
    for r, (a, b) in enumerate(tok.merges_):
        tok.vocab_[256 + r] = tok.vocab_[a] + tok.vocab_[b]
    return tok


def truncated_bpe(tok, vocab_size):
    """A copy of a fitted BPETokenizer without special token that keeps only its first vocab_size - n_base_ merges:
    the tokenizer that fit would have learned with this vocab_size (merges are learned one after the other, the same
    first ones whatever the final size, as long as min_frequency does not stop the training)."""
    if tok.special_ids_:
        raise ValueError("truncated_bpe: only for a tokenizer without special token")
    k = vocab_size - tok.n_base_
    if not 0 <= k <= len(tok.merges_):
        raise ValueError(f"truncated_bpe: vocab_size must be between {tok.n_base_} and "
                         f"{tok.n_base_ + len(tok.merges_)}, got {vocab_size}")
    small = type(tok)(vocab_size=vocab_size, level=tok.level, pre_tokenizer=tok.pre_tokenizer,
                      min_frequency=tok.min_frequency)
    small.n_base_, small.merges_, small.special_ids_ = tok.n_base_, list(tok.merges_[:k]), {}
    small.vocab_ = {i: tok.vocab_[i] for i in range(tok.n_base_ + k)}
    return small


print(f"Holmes : {len(HOLMES_TRAIN):_} caractères d'entraînement, {len(HOLMES_VAL):_} de validation ; Verne : "
      f"{len(VERNE_TRAIN):_} et {len(VERNE_VAL):_} ; extraits de {len(EXCERPT_H):_} caractères.".replace("_", " "))

Holmes : 505 966 caractères d'entraînement, 56 237 de validation ; Verne : 379 133 et 42 203 ; extraits de 50 000 caractères.


### Ex B2.13 — Tokenizers réels : GPT-2, BERT et un tokenizer multilingue sur Holmes et Verne 📦 ★★ ⏱️ 20 min
**Objectif :** charger trois tokenizers réels à une révision fixée, et mesurer ce que leur coûte un texte anglais et un texte français.  
**Prérequis :** ch. 22 (vocabulaire et encodage) · fiche §202.1, §202.5 (« Utiliser un tokenizer avec `transformers` »), §202.6 (« Deux rapports ») · **Fil rouge :** Holmes/Verne · **Parcours :** R, C

**Contrat**
- `load_tokenizers_13()` : un dictionnaire de trois tokenizers de `transformers`, chacun chargé par `AutoTokenizer.from_pretrained(dépôt, revision=révision)` avec le dépôt et la révision de `TOKENIZER_REPOS` :
  - `"gpt2"`, le BPE octet de GPT-2 ;
  - `"bert"`, le WordPiece de `bert-base-uncased` ;
  - `"e5"`, le tokenizer multilingue de `multilingual-e5-small` (celui de XLM-R).
- `measure_13(tok, text)` : un dictionnaire de quatre mesures :
  - `"tokens"` : le nombre de tokens du texte **seul**, sans les tokens spéciaux que certains tokenizers ajoutent autour d'un texte ;
  - `"words"` : le nombre de mots, `len(text.split())` ;
  - `"fertility"` : tokens / mots ;
  - `"bytes_per_token"` : octets UTF-8 du texte / tokens, la compression.

La cellule de vérification compare tes trois tokenizers à ceux de référence (`wbtext.load_tokenizer`) : leur type, leur taille et les identifiants d'une phrase d'essai. Elle mesure ensuite chacun sur `EXCERPT_H` et `EXCERPT_V` et affiche le tableau des six lignes. Quatre valeurs sont vérifiées, calculées par ta fonction (rien à taper) :
- a) le nombre de tokens de GPT-2 sur l'extrait de Holmes ;
- b) celui de BERT sur l'extrait de Verne ;
- c) la fertilité de e5 sur Holmes ;
- d) la compression de e5 sur Verne.

Dans tes notes : quel tokenizer est le plus économe sur Holmes, et sur Verne ? BERT *uncased* paraît économe sur Verne : `bert.decode(bert(phrase, add_special_tokens=False)["input_ids"])` redonne-t-il une phrase accentuée de Verne ? Que perd-il ? Affiche les tokens d'une même phrase avec les trois tokenizers (`tok.tokenize(phrase)`) : que signalent « Ġ », « ## » et « ▁ » ?

In [25]:
from transformers import AutoTokenizer


def load_tokenizers_13():
    """{"gpt2": ..., "bert": ..., "e5": ...}: the AutoTokenizer of each repository of TOKENIZER_REPOS, at its revision."""
    raise NotImplementedError("load_tokenizers_13")


def measure_13(tok, text):
    """{"tokens": int, "words": int, "fertility": float, "bytes_per_token": float} for one tokenizer and one text."""
    raise NotImplementedError("measure_13")

In [26]:
REF_13 = {name: wbtext.try_tokenizer(name, "B2.13") for name in ("gpt2", "bert", "e5")}
KEYS_13 = ("tokens", "words", "fertility", "bytes_per_token")
rows_13 = {}
if any(tok is None for tok in REF_13.values()):
    print("⏳ Ex B2.13 : sans les tokenizers de référence, la vérification attend une connexion au Hub (ou Colab).")
else:
    with checked("B2.13"):
        toks_13 = load_tokenizers_13()
        if not isinstance(toks_13, dict) or set(toks_13) != set(REF_13):
            got = sorted(toks_13) if isinstance(toks_13, dict) else type(toks_13).__name__
            print(f"❌ Ex B2.13 : load_tokenizers_13 doit rendre un dictionnaire de clés 'gpt2', 'bert' et 'e5' ; reçu {got}.")
            raise StopCheck
        probe_13 = "Phileas Fogg paria 20 000 livres, déjà ! 🙂"
        for name, ref in REF_13.items():
            tok = toks_13[name]
            ok = (callable(tok) and type(tok).__name__ == type(ref).__name__ and len(tok) == len(ref)
                  and tok(probe_13)["input_ids"] == ref(probe_13)["input_ids"])
            verdict("B2.13", ok, f"{name} : {type(ref).__name__}, {len(ref):_} tokens, mêmes ids que la référence.".replace("_", " "),
                    f"{name} : ce n'est pas le tokenizer de {TOKENIZER_REPOS[name][0]} (type, taille du vocabulaire ou ids "
                    "d'une phrase d'essai différents de la référence).")
        for name in REF_13:
            for label, text in (("Holmes", EXCERPT_H), ("Verne", EXCERPT_V)):
                m = measure_13(toks_13[name], text)
                if not returned("B2.13", "measure_13", m):
                    raise StopCheck
                if not isinstance(m, dict) or any(key not in m for key in KEYS_13):
                    print(f"❌ Ex B2.13 : measure_13 doit rendre un dictionnaire de clés {', '.join(KEYS_13)} ; reçu "
                          f"{sorted(m) if isinstance(m, dict) else type(m).__name__}.")
                    raise StopCheck
                rows_13[(name, label)] = m
        table_13 = pd.DataFrame({f"{name} · {label}": m for (name, label), m in rows_13.items()}).T
        print(table_13.to_string(float_format=lambda x: f"{x:.3f}"))
        m = rows_13[("gpt2", "Holmes")]
        verdict("B2.13", m["words"] == len(EXCERPT_H.split()) and math.isclose(m["fertility"], m["tokens"] / m["words"])
                and math.isclose(m["bytes_per_token"], n_bytes(EXCERPT_H) / m["tokens"]),
                "les quatre mesures sont cohérentes (mots de split(), fertilité = tokens / mots, octets UTF-8 / tokens).",
                "relis le contrat : words = len(text.split()), fertility = tokens / words, bytes_per_token = octets UTF-8 "
                "du texte / tokens.")
        wb.check("B2.13a", rows_13[("gpt2", "Holmes")]["tokens"], computed=True)
        wb.check("B2.13b", rows_13[("bert", "Verne")]["tokens"], computed=True)
        wb.check("B2.13c", rows_13[("e5", "Holmes")]["fertility"], computed=True)
        wb.check("B2.13d", rows_13[("e5", "Verne")]["bytes_per_token"], computed=True)

⏳ Ex B2.13 : pas encore fait (load_tokenizers_13).


### Ex B2.14 — Prédire : quel texte coûte le plus de tokens, Holmes, Verne, des nombres ou des emoji ? 🔮 ★ ⏱️ 10 min
**Objectif :** prédire, en ordres de grandeur, ce que coûtent en tokens un texte anglais, un texte français, des nombres et des emoji.  
**Prérequis :** Ex B2.13 · fiche §202.2 (octets UTF-8), §202.4, §202.5 (WordPiece), §202.6 (« Les chiffres et les lettres ») · **Fil rouge :** Holmes/Verne · **Parcours :** R, C

La cellule suivante prépare deux textes de plus, à côté des deux extraits de B2.13 :
- `NUMBERS_14` : 2 000 nombres de 1 à 5 chiffres, séparés par des espaces (11 768 octets) ;
- `EMOJI_14` : 1 200 emoji collés, sans espace (6 000 octets ; « 👍🏽 » et « ❤️ » font deux points de code chacun).

L'expérience compte les tokens des quatre textes, tokens spéciaux exclus, avec six tokenizers : GPT-2, BERT, e5, CamemBERT, Qwen3 et SmolLM2. **Avant d'exécuter**, écris tes trois prédictions :
- a) `costliest_14` : avec GPT-2, le texte qui a le **moins** d'octets par token (celui qui coûte le plus cher pour sa taille) : `"A"` Holmes, `"B"` Verne, `"C"` les nombres, `"D"` les emoji ;
- b) `ratio_14` : avec CamemBERT, le rapport entre les tokens de `EXCERPT_V` et ceux de `EXCERPT_H`, deux textes de 50 000 caractères : `"A"` moins de 0,9, `"B"` de 0,9 à 1,1, `"C"` plus de 1,1 ;
- c) `bert_emoji_14` : le nombre de tokens de BERT (`bert-base-uncased`) pour `EMOJI_14` : `"A"` de 1 à 10, `"B"` de 11 à 2 000, `"C"` plus de 2 000.

La cellule d'expérience n'affiche rien tant que tes trois prédictions ne sont pas écrites ; elle les vérifie à la fin.

In [27]:
# Two more texts of about 6 000 to 12 000 bytes, next to the two excerpts of B2.13
NUMBERS_14 = " ".join(str(i * 7919 % 100000) for i in range(2000))    # 2 000 numbers of 1 to 5 digits, spaced
EMOJI_14 = "🙂🚀🎉👍🏽🐍❤️" * 200                                        # 1 200 emoji in a row, no space
TEXTS_14 = {"Holmes": EXCERPT_H, "Verne": EXCERPT_V, "nombres": NUMBERS_14, "emoji": EMOJI_14}
print({name: f"{len(text):_} points de code, {n_bytes(text):_} octets".replace("_", " ") for name, text in TEXTS_14.items()})

{'Holmes': '50 000 points de code, 50 980 octets', 'Verne': '50 000 points de code, 51 497 octets', 'nombres': '11 768 points de code, 11 768 octets', 'emoji': '1 600 points de code, 6 000 octets'}


📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [28]:
costliest_14 = ...    # a) with GPT-2, the text with the FEWEST bytes per token:
                      #    "A" Holmes, "B" Verne, "C" the numbers, "D" the emoji
ratio_14 = ...        # b) with CamemBERT, tokens of EXCERPT_V / tokens of EXCERPT_H (the same 50 000 characters):
                      #    "A" under 0.9, "B" from 0.9 to 1.1, "C" over 1.1
bert_emoji_14 = ...   # c) the number of tokens of BERT (bert-base-uncased) for EMOJI_14, special tokens left out:
                      #    "A" from 1 to 10, "B" from 11 to 2 000, "C" over 2 000

In [29]:
if any(answer is ... or answer is None for answer in [costliest_14, ratio_14, bert_emoji_14]):
    print("⏳ Ex B2.14 : écris d'abord tes trois prédictions dans la cellule précédente.")
else:
    names_14 = ("gpt2", "bert", "e5", "camembert", "qwen3", "smollm2")
    toks_14 = {name: wbtext.try_tokenizer(name, "B2.14") for name in names_14}
    counts_14 = {name: {label: wbtext.count_tokens(tok, text) for label, text in TEXTS_14.items()}
                 for name, tok in toks_14.items() if tok is not None}
    if any(name not in counts_14 for name in ("gpt2", "bert", "camembert")):
        print("⏳ Ex B2.14 : l'expérience attend les tokenizers de GPT-2, de BERT et de CamemBERT (connexion au Hub, ou Colab).")
    else:
        tokens_14 = pd.DataFrame(counts_14).T
        per_token_14 = pd.DataFrame({name: {label: n_bytes(TEXTS_14[label]) / n for label, n in row.items()}
                                     for name, row in counts_14.items()}).T
        print("Nombre de tokens (tokens spéciaux exclus) :")
        print(tokens_14.to_string())
        print("\nOctets par token :")
        print(per_token_14.to_string(float_format=lambda x: f"{x:.2f}"))
        print("\nTokens de Verne / tokens de Holmes :", ", ".join(f"{name} {fr(row['Verne'] / row['Holmes'], 2)}"
                                                             for name, row in counts_14.items()))
        print("BERT sur les emoji :", toks_14["bert"].convert_ids_to_tokens(toks_14["bert"](EMOJI_14, add_special_tokens=False)["input_ids"])[:5])
        for id_14, value_14 in (("B2.14a", costliest_14), ("B2.14b", ratio_14), ("B2.14c", bert_emoji_14)):
            wb.check(id_14, value_14)

⏳ Ex B2.14 : écris d'abord tes trois prédictions dans la cellule précédente.


Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Compare les six tokenizers texte par texte : lesquels traitent les chiffres un par un (fiche §202.6), et que change ce choix pour les nombres ? Sur quel texte les six tokenizers sont-ils le plus inégaux, et pourquoi ?

## Partie B · Ton BPE : tokenizer.py, puis un tokenizer à réparer et la comparaison à tokenizers

Tu écris dans `mylearn/tokenizer.py` un BPE complet, au niveau caractère et au niveau octet : compter et fusionner les paires (B2.15), apprendre les fusions (B2.16), encoder par rang et décoder sans perte (B2.17). Tu répares ensuite un petit tokenizer qui abîme les accents (B2.18), puis tu compares ton BPE à la bibliothèque `tokenizers`, écrite en Rust : mêmes fusions, autre vitesse (B2.19). Ton `BPETokenizer` resservira en B3 (un mini-GPT sur des tokens BPE).

**Contrat commun de `tokenizer.py`** (les docstrings, plus ces précisions que vérifient les tests) :
- les identifiants : d'abord l'alphabet, `0 … n_base_ − 1` (les 256 octets au niveau octet, ou les caractères des pré-tokens triés par point de code au niveau caractère) ; puis une fusion par identifiant, dans l'ordre d'apprentissage (la fusion de rang $r$ crée l'identifiant `n_base_ + r`) ; puis les tokens spéciaux ;
- tes fonctions ne modifient jamais leurs entrées ; une entrée invalide lève `ValueError`, jamais `KeyError` ni `AttributeError` ;
- la bibliothèque standard seulement ; `pre_tokenize`, `save` et `load` sont fournis, déjà écrits.

### Ex B2.15 — get_pair_counts et merge_pair 🔨 ★★ ⏱️ 20 min
**Objectif :** compter les paires de tokens voisins d'un corpus pondéré, puis fusionner une paire dans une suite d'identifiants.  
**Prérequis :** ✏️ B2.3 · fiche §202.3 (« L'entraînement », « Ta librairie : `tokenizer.py` ») · **Fil rouge :** Holmes/Verne · **mylearn :** `tokenizer.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/tokenizer.py` (créé par `python tools/start_chapter.py B2`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. La bibliothèque standard seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb2_tokenizer.py -q` les lance tous dans un terminal.

**Contrat**
- `get_pair_counts(words)` :
  - `words` est un corpus pré-tokenisé, `{mot en tuple d'identifiants: fréquence}` ; tout mapping convient, en lecture seule (les tests en passent un `MappingProxyType` et un `Counter`), et il n'est pas modifié ;
  - rend un `dict` (un `Counter` convient) `{(gauche, droite): compte}` des paires d'identifiants voisins, chaque occurrence comptant la fréquence de son mot ;
  - les paires qui se chevauchent comptent : `(1, 1, 1)` contient deux fois `(1, 1)` ; aucune paire ne franchit la frontière de deux mots ;
  - une fréquence négative lève `ValueError` ; un mot de fréquence 0 n'est pas testé (le plus naturel : il ne produit aucune paire).
- `merge_pair(ids, pair, new_id)` :
  - remplace chaque occurrence de `pair` par `new_id`, de gauche à droite et sans chevauchement (comme `str.replace`) ;
  - rend toujours une **liste neuve** : jamais l'entrée elle-même, même sans aucune occurrence, et jamais un tuple ;
  - accepte un tuple, sans le modifier ; une paire qui n'a pas exactement deux identifiants lève `ValueError`.

La première cellule de vérification essaie tes deux fonctions sur des cas visibles, puis compte les paires des mots de l'extrait de Verne (chaque caractère écrit par son point de code) et les compare à un comptage direct ; elle essaie enfin deux refus. La seconde lance les 42 tests de l'exercice (`-k "test_get_pair_counts_ or test_merge_pair_"`).

Dans tes notes : l'extrait de Verne a 7 948 mots, mais combien de mots distincts (la cellule l'affiche) ? Pourquoi compter les paires sur les mots distincts, avec leur fréquence, plutôt que sur le texte entier ?

In [30]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")   # reload your saved file
with checked("B2.15"):
    T = mylearn.tokenizer
    corpus_15 = {(1, 2, 3): 2, (1, 2): 1, (4, 4, 4): 3, (5,): 7}
    arg_15 = dict(corpus_15)                                     # your function receives a copy
    counts_15 = T.get_pair_counts(MappingProxyType(arg_15))      # a read-only view, as in the tests
    if returned("B2.15", "get_pair_counts", counts_15):
        verdict("B2.15", isinstance(counts_15, dict) and dict(counts_15) == {(1, 2): 3, (2, 3): 2, (4, 4): 6},
                "get_pair_counts : (1, 2) 3 fois, (2, 3) 2 fois, (4, 4) 6 fois ; un mot d'un seul id n'a pas de paire.",
                f"get_pair_counts({corpus_15}) doit rendre {{(1, 2): 3, (2, 3): 2, (4, 4): 6}} (chaque paire voisine, "
                f"pondérée par la fréquence du mot, chevauchements compris) ; reçu {counts_15!r}.")
        verdict("B2.15", arg_15 == corpus_15, "le corpus reçu n'a pas changé.", "get_pair_counts a modifié le corpus reçu.")
    words_15 = Counter(tuple(map(ord, w)) for w in EXCERPT_V.split())      # the words of Verne, as code points
    expected_15 = Counter()
    for w, f in words_15.items():
        for p in zip(w, w[1:]):
            expected_15[p] += f
    got_15 = T.get_pair_counts(dict(words_15))
    verdict("B2.15", isinstance(got_15, dict) and dict(got_15) == dict(expected_15),
            f"sur les {len(words_15):_} mots distincts de l'extrait de Verne : {len(expected_15):_} paires, les mêmes comptes "
            "qu'un comptage direct.".replace("_", " "),
            "sur les mots de l'extrait de Verne, tes comptes diffèrent d'un comptage direct : as-tu pondéré par la "
            "fréquence et gardé les paires qui se chevauchent ?")
    refuses("B2.15", "get_pair_counts({(1, 2): -1}) (une fréquence négative)", lambda: T.get_pair_counts({(1, 2): -1}))

    ids_15 = (3, 1, 2, 1, 2, 1, 1, 1)                            # a tuple: merge_pair must accept it
    merged_15 = T.merge_pair(ids_15, (1, 2), 9)
    if returned("B2.15", "merge_pair", merged_15):
        verdict("B2.15", type(merged_15) is list and merged_15 == [3, 9, 9, 1, 1, 1],
                "merge_pair((3, 1, 2, 1, 2, 1, 1, 1), (1, 2), 9) = [3, 9, 9, 1, 1, 1], une list.",
                f"merge_pair({ids_15}, (1, 2), 9) doit rendre la list [3, 9, 9, 1, 1, 1] ; reçu {merged_15!r}.")
    overlap_15 = T.merge_pair([1, 1, 1], (1, 1), 5)
    verdict("B2.15", overlap_15 == [5, 1], "[1, 1, 1] avec (1, 1) donne [5, 1] : sans chevauchement, de gauche à droite.",
            f"merge_pair([1, 1, 1], (1, 1), 5) doit rendre [5, 1] (sans chevauchement, de gauche à droite) ; reçu {overlap_15!r}.")
    same_15 = [4, 5]
    out_15 = T.merge_pair(same_15, (1, 2), 9)
    verdict("B2.15", out_15 == [4, 5] and out_15 is not same_15 and same_15 == [4, 5],
            "sans occurrence de la paire : une liste neuve égale à l'entrée, l'entrée intacte.",
            "sans occurrence de la paire, merge_pair doit rendre une liste neuve (pas l'entrée elle-même), égale à "
            f"l'entrée, et laisser l'entrée intacte ; reçu {out_15!r}, entrée {same_15!r}.")
    refuses("B2.15", "merge_pair([1, 2], (1, 2, 3), 9) (une paire de trois ids)", lambda: T.merge_pair([1, 2], (1, 2, 3), 9))

ℹ️ Ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2.
⏳ Ex B2.15 : pas encore fait (ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2).


In [31]:
if written(lambda: mylearn.tokenizer.get_pair_counts({(1, 2): 1})):
    run_pytest_group(TEST_TOK, "test_get_pair_counts_ or test_merge_pair_")   # 42 tests
else:
    print("⏳ Ex B2.15 : les tests attendent tes fonctions.")

⏳ Ex B2.15 : les tests attendent tes fonctions.


### Ex B2.16 — BPETokenizer.fit : apprendre les fusions, au niveau caractère puis octet 🔨 ★★★ ⏱️ 50 min
**Objectif :** apprendre les fusions d'un BPE au niveau caractère et au niveau octet, avec le départage et les conditions d'arrêt de `tokenizers`.  
**Prérequis :** Ex B2.15 · ✏️ B2.3 · fiche §202.3 (encadré 🕰️ sur le départage), §202.4 (tokens spéciaux) · **Fil rouge :** Holmes/Verne · **mylearn :** `tokenizer.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/tokenizer.py` (créé par `python tools/start_chapter.py B2`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. La bibliothèque standard seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb2_tokenizer.py -q` les lance tous dans un terminal.

**Contrat** de `BPETokenizer.fit(texts)` :
- **Entrée et sortie** : il rend `self` et pose les quatre attributs `n_base_`, `merges_`, `vocab_` et `special_ids_`. Une seule `str` est **un** texte ; tout itérable de `str` convient, un générateur aussi, et la liste reçue n'est pas modifiée.
- **Corpus** : chaque texte est découpé par `pre_tokenize(texte, self.pre_tokenizer, tokens spéciaux)`. Les tokens spéciaux, `unk_token` compris, sont **retirés avant tout comptage** : leurs caractères n'entrent pas dans l'alphabet, et les fusions sont apprises sur le reste. Des pré-tokens identiques sont comptés une fois, avec leur fréquence.
- **Alphabet** : au niveau octet, `n_base_ = 256` et `vocab_[i] = bytes([i])`. Au niveau caractère, les caractères distincts des pré-tokens (sans l'espace avec `'whitespace'`), triés par point de code ; `vocab_[i]` contient leurs octets UTF-8.
- **Fusions** : chaque étape fusionne la paire la plus fréquente, et à égalité la plus petite paire `(gauche, droite)`. La fusion de rang $r$ crée l'identifiant `n_base_ + r`, avec `vocab_[n_base_ + r] = vocab_[a] + vocab_[b]`. `merges_` est une liste de **tuples** `(a, b)`, dans l'ordre d'apprentissage.
- **Arrêt** : l'entraînement s'arrête quand `n_base_ + fusions + spéciaux` atteint `vocab_size`, quand il ne reste aucune paire, ou quand la meilleure paire apparaît moins de `min_frequency` fois (`min_frequency = 0` est permis). `vocab_size` compte **tous** les tokens spéciaux, `unk_token` compris quand il est ajouté : avec `vocab_size = n_base_ + spéciaux`, aucune fusion.
- **Tokens spéciaux** : `special_ids_` les place après les fusions, dans l'ordre donné, puis `unk_token` s'il n'y est pas déjà (une seule fois). `vocab_` contient aussi leurs octets UTF-8, donc `len(vocab_) = n_base_ + fusions + spéciaux`. Un token spécial donné deux fois n'est pas testé.
- **Erreurs** : `ValueError` pour un `level` ou un `pre_tokenizer` inconnu, un `unk_token` au niveau octet, et `vocab_size < n_base_ +` spéciaux (au niveau caractère, une fois le corpus lu).

Les tests comparent tes fusions à celles de `tokenizers` aux deux niveaux, sur des extraits de Holmes et de Verne. Une version qui recompte toutes les paires à chaque fusion les passe en quelques secondes ; sur un roman entier, elle met de 30 s à 2 min pour 2 000 tokens, que tu sentiras en B2.19 (l'indice 2 dit comment aller plus vite).

La première cellule de vérification essaie `fit` sur trois petites phrases, puis compare tes 400 tokens de l'extrait de Verne à ceux de `BpeTrainer`, essaie le niveau octet avec deux tokens spéciaux, et trois refus. La seconde lance les 49 tests de l'exercice (`-k "test_pre_tokenize_ or test_bpe_fit_"`, dont les 5 de `pre_tokenize`, fourni).

Dans tes notes : pourquoi l'alphabet est-il trié par point de code, et que deviendrait la comparaison à `tokenizers` s'il suivait l'ordre d'apparition des caractères ? Pourquoi `vocab_size` compte-t-il les tokens spéciaux (pense à la table d'embedding d'un modèle, §202.7) ?

In [32]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")   # reload your saved file
with checked("B2.16"):
    T = mylearn.tokenizer
    texts_16 = ["le chat dort, le rat court", "les chats et les rats", "un chaton"]
    copy_16 = list(texts_16)
    tok_16 = T.BPETokenizer(vocab_size=24, level="char", min_frequency=1)
    out_16 = tok_16.fit(copy_16)
    verdict("B2.16", out_16 is tok_16, "fit rend self.", "fit doit rendre self (return self à la fin).")
    verdict("B2.16", copy_16 == texts_16, "la liste des textes n'a pas changé.", "fit a modifié la liste des textes reçue.")
    alphabet_16 = sorted(set("".join(texts_16)) - {" "})
    got_alphabet_16 = [tok_16.vocab_[i] for i in range(getattr(tok_16, "n_base_", 0))]
    verdict("B2.16", got_alphabet_16 == [c.encode() for c in alphabet_16],
            f"alphabet : les {len(alphabet_16)} caractères des pré-tokens, triés par point de code (« , » en premier).",
            f"l'alphabet doit être les caractères des pré-tokens (sans les espaces), triés par point de code : attendu "
            f"{''.join(alphabet_16)!r}, reçu {b''.join(got_alphabet_16)!r}.")
    merges_16 = getattr(tok_16, "merges_", None)
    verdict("B2.16", isinstance(merges_16, list) and all(type(m) is tuple for m in merges_16)
            and len(tok_16.vocab_) == tok_16.n_base_ + len(merges_16) + len(tok_16.special_ids_) <= 24,
            f"merges_ : une liste de {len(merges_16 or [])} tuples ; len(vocab_) = n_base_ + fusions + spéciaux ≤ vocab_size.",
            "merges_ doit être une liste de tuples (a, b), et vocab_ contenir l'alphabet, les fusions et les tokens "
            "spéciaux, vocab_size au plus.")
    # the same merges as BpeTrainer of tokenizers, on the Verne excerpt
    hf_16 = reference_char_bpe(EXCERPT_V, 400)
    mine_16 = T.BPETokenizer(vocab_size=400, level="char").fit(EXCERPT_V)
    want_16 = [hf_16.id_to_token(i) for i in range(hf_16.get_vocab_size())]
    got_16 = [mine_16.vocab_[i].decode("utf-8", errors="replace") for i in range(len(mine_16.vocab_))]
    first_16 = next((i for i, (a, b) in enumerate(zip(got_16, want_16)) if a != b), None)
    verdict("B2.16", got_16 == want_16,
            f"extrait de Verne, niveau caractère, 400 tokens : les mêmes {len(want_16)} tokens, dans le même ordre, que "
            "BpeTrainer de tokenizers.",
            (f"extrait de Verne, 400 tokens : tes tokens diffèrent de ceux de BpeTrainer dès l'id {first_16} (toi "
             f"{got_16[first_16]!r}, tokenizers {want_16[first_16]!r}) : relis le départage et le tri de l'alphabet."
             if first_16 is not None else f"extrait de Verne : {len(got_16)} tokens chez toi, {len(want_16)} chez "
             "BpeTrainer : relis les conditions d'arrêt (vocab_size, min_frequency)."))
    # byte level, special tokens
    byte_16 = T.BPETokenizer(vocab_size=256 + 10 + 2, pre_tokenizer="gpt2", special_tokens=["<|endoftext|>", "<pad>"])
    byte_16.fit(EXCERPT_V)
    verdict("B2.16", byte_16.n_base_ == 256 and all(byte_16.vocab_[i] == bytes([i]) for i in range(256))
            and len(byte_16.merges_) == 10 and byte_16.special_ids_ == {"<|endoftext|>": 266, "<pad>": 267},
            "niveau octet : 256 octets, 10 fusions, puis <|endoftext|> en 266 et <pad> en 267 (vocab_size compte les spéciaux).",
            "niveau octet avec vocab_size = 268 et deux tokens spéciaux : attendu n_base_ = 256 (vocab_[i] = bytes([i])), "
            f"10 fusions, special_ids_ = {{'<|endoftext|>': 266, '<pad>': 267}} ; reçu n_base_ = "
            f"{getattr(byte_16, 'n_base_', None)}, {len(getattr(byte_16, 'merges_', []))} fusions, "
            f"special_ids_ = {getattr(byte_16, 'special_ids_', None)}.")
    refuses("B2.16", "level='word'", lambda: T.BPETokenizer(level="word").fit("abc"))
    refuses("B2.16", "unk_token au niveau octet", lambda: T.BPETokenizer(unk_token="<unk>").fit("abc"))
    refuses("B2.16", "vocab_size = 100 au niveau octet (moins que les 256 octets)", lambda: T.BPETokenizer(vocab_size=100).fit("abc"))

ℹ️ Ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2.
⏳ Ex B2.16 : pas encore fait (ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2).


In [33]:
if written(lambda: mylearn.tokenizer.BPETokenizer(vocab_size=258).fit("aa aa")):
    run_pytest_group(TEST_TOK, "test_pre_tokenize_ or test_bpe_fit_")   # 49 tests
else:
    print("⏳ Ex B2.16 : les tests attendent ta méthode fit.")

⏳ Ex B2.16 : les tests attendent ta méthode fit.


### Ex B2.17 — encode et decode : fusions appliquées par rang, aller-retour sans perte 🔨 ★★★ ⏱️ 40 min
**Objectif :** encoder un texte en appliquant les fusions par rang, puis le décoder sans perte au niveau octet.  
**Prérequis :** Ex B2.16 · ✏️ B2.4 · fiche §202.3 (« Encoder un mot nouveau : les fusions par rang »), §202.4 · **Fil rouge :** Holmes/Verne · **mylearn :** `tokenizer.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/tokenizer.py` (créé par `python tools/start_chapter.py B2`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. La bibliothèque standard seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb2_tokenizer.py -q` les lance tous dans un terminal.

**Contrat**
- `encode(text)` rend une `list` d'`int` (`[]` pour un texte vide) :
  - toute clé de `special_ids_` écrite dans le texte devient son identifiant, `unk_token` compris : « <unk> » tapé tel quel donne l'identifiant de `<unk>` ;
  - dans chaque autre pré-token, jamais entre deux, on part des identifiants de base (octets UTF-8, ou caractères), puis on fusionne la paire voisine de **plus petit rang** parmi celles de `merges_`, et ainsi de suite jusqu'à ce qu'aucune paire voisine ne soit une fusion apprise ;
  - au niveau caractère, chaque caractère inconnu donne l'identifiant de `unk_token`, un par caractère.
- `decode(ids)` recolle les octets de tous les tokens, **puis** décode l'UTF-8 avec `errors="replace"` : `[0xC3, 0xA9]` donne « é », et `[0xC3]` seul donne « � ». Il accepte tout entier, Python ou NumPy, qui est une clé de `vocab_`. Avec `'whitespace'`, les espaces disparaissent.
- `tokenize(text)` rend les tokens de `encode(text)` en `str`, « � » pour un morceau de caractère.
- `ValueError`, jamais `AttributeError` ni `KeyError`, pour :
  - un tokenizer non entraîné, dans `encode` comme dans `decode` ;
  - un caractère inconnu sans `unk_token` ;
  - un identifiant inconnu passé à `decode`, −1 compris (ce n'est pas un indice Python).
- `encode` et `decode` ne lisent que les hyperparamètres et les quatre attributs appris : un cache privé posé par `fit` casse le tokenizer rechargé par `load` (testé).

Les tests comparent tes identifiants à ceux de `tokenizers`, vérifient l'aller-retour sans perte sur environ 300 chaînes Unicode tirées au hasard (emoji, idéogrammes, accents combinants, espaces multiples), et que ton nombre de tokens reste à moins de 2 % de celui d'un BPE `ByteLevel` de même taille.

La première cellule de vérification compare ton encodage d'un passage de Verne à celui de `tokenizers`, essaie `<unk>`, l'aller-retour d'un texte accentué avec un emoji et des idéogrammes, un octet seul, `tokenize`, et quatre refus. La seconde lance les 36 tests de l'exercice (`-k "test_bpe_encode_ or test_bpe_decode_ or test_bpe_tokenize_ or test_bpe_save_load_"`).

Dans tes notes : avec `'whitespace'`, que redonne `decode(encode("le chat dort"))`, et pourquoi le pré-tokenizer `'gpt2'` règle-t-il ce problème ? Que rend `tokenize("été")` au niveau octet avec seulement 256 tokens, et pourquoi ?

In [34]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")   # reload your saved file
with checked("B2.17"):
    T = mylearn.tokenizer
    char_17 = T.BPETokenizer(vocab_size=400, level="char").fit(EXCERPT_V)
    hf_17 = reference_char_bpe(EXCERPT_V, 400)
    sample_17 = EXCERPT_V[20000:24000]                    # characters seen during fit
    ids_17 = char_17.encode(sample_17)
    if returned("B2.17", "encode", ids_17):
        verdict("B2.17", type(ids_17) is list and all(isinstance(i, int) for i in ids_17) and ids_17 == hf_17.encode(sample_17).ids,
                f"niveau caractère : les {len(ids_17)} ids d'un passage de Verne sont ceux de tokenizers (fusions par rang).",
                "niveau caractère : encode doit rendre une list d'int identique aux ids de tokenizers (BpeTrainer) ; "
                "applique-t-il les fusions par rang, dans chaque pré-token ?")
    verdict("B2.17", char_17.encode("") == [], "encode('') = [].", f"encode('') doit rendre [] ; reçu {char_17.encode('')!r}.")
    unk_17 = T.BPETokenizer(vocab_size=300, level="char", unk_token="<unk>").fit(EXCERPT_V)
    u_17 = unk_17.special_ids_.get("<unk>")
    got_unk_17 = unk_17.encode("Fogg 🙂🙂 <unk>")
    verdict("B2.17", u_17 is not None and got_unk_17[-3:] == [u_17] * 3 and got_unk_17.count(u_17) == 3,
            "niveau caractère : chaque caractère inconnu donne l'id de <unk>, et « <unk> » écrit dans le texte aussi.",
            f"« Fogg 🙂🙂 <unk> » : attendu trois fois l'id {u_17} de <unk> (un par emoji inconnu, un pour « <unk> » "
            f"écrit tel quel) ; reçu {got_unk_17!r}.")
    byte_17 = T.BPETokenizer(vocab_size=600, pre_tokenizer="gpt2").fit(EXCERPT_V)
    text_17 = VERNE_VAL[:3000] + " 🙂 漢字, é !"
    try:
        back_17 = byte_17.decode(byte_17.encode(text_17))
    except UnicodeDecodeError as exc:
        back_17 = None
        print(f"❌ Ex B2.17 : decode(encode(texte)) lève UnicodeDecodeError ({exc.reason}) : decode doit recoller les "
              "octets de tous les tokens, puis décoder l'UTF-8 avec errors='replace'.")
    if back_17 is not None:
        at_17 = next((i for i, (a, b) in enumerate(zip(back_17, text_17)) if a != b), min(len(back_17), len(text_17)))
        verdict("B2.17", back_17 == text_17,
                "niveau octet, pré-tokenizer 'gpt2' : decode(encode(texte)) redonne le texte exact (emoji et idéogrammes compris).",
                "niveau octet avec 'gpt2', decode(encode(texte)) doit redonner le texte exact ; à partir du caractère "
                f"{at_17}, attendu {text_17[at_17:at_17 + 12]!r}, reçu {back_17[at_17:at_17 + 12]!r}.")
    try:
        lone_17 = byte_17.decode([0xC3])
        verdict("B2.17", lone_17 == "�", "decode([0xC3]) (un octet seul) = '�'.",
                f"decode([0xC3]), le premier octet de « é » seul, doit rendre '�' ; reçu {lone_17!r}.")
    except UnicodeDecodeError as exc:
        print(f"❌ Ex B2.17 : decode([0xC3]), le premier octet de « é » seul, lève UnicodeDecodeError ({exc.reason}) au "
              "lieu de rendre '�' (errors='replace').")
    pieces_17 = byte_17.tokenize("été")
    verdict("B2.17", isinstance(pieces_17, list) and all(isinstance(p, str) for p in pieces_17) and "".join(pieces_17) == "été",
            f"tokenize('été') = {pieces_17!r}.", f"tokenize doit rendre une liste de str ; reçu {pieces_17!r}.")
    refuses("B2.17", "encode avant fit", lambda: T.BPETokenizer().encode("abc"))
    refuses("B2.17", "decode([-1])", lambda: byte_17.decode([-1]))
    refuses("B2.17", "decode([len(vocab_)])", lambda: byte_17.decode([len(byte_17.vocab_)]))
    refuses("B2.17", "niveau caractère sans unk_token : encode('🙂')", lambda: char_17.encode("🙂"))

ℹ️ Ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2.
⏳ Ex B2.17 : pas encore fait (ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2).


In [35]:
if written(lambda: mylearn.tokenizer.BPETokenizer(vocab_size=258).fit("aa aa").encode("aa")):
    run_pytest_group(TEST_TOK, "test_bpe_encode_ or test_bpe_decode_ or test_bpe_tokenize_ or test_bpe_save_load_")   # 36 tests
else:
    print("⏳ Ex B2.17 : les tests attendent tes méthodes fit et encode.")

⏳ Ex B2.17 : les tests attendent tes méthodes fit et encode.


### Ex B2.18 — Le tokenizer qui abîme les accents : trois bugs d'Unicode et de fusions 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver et corriger les trois erreurs d'un petit tokenizer BPE, guidé par des diagnostics enchaînés.  
**Prérequis :** Ex B2.17 · fiche §202.2 (normalisation), §202.3, §202.4 · tableau des pièges ⚠️ · **Fil rouge :** Verne · **Parcours :** R, C

Un collègue a écrit un petit tokenizer BPE au niveau octet, avec les pré-tokens de GPT-2, en cinq fonctions : `clean_18`, `merge_18`, `train_18`, `encode_18` et `decode_18`. Il tourne, mais il abîme le français. Le code contient **trois** erreurs. Corrige-le **sur place**, dans sa cellule, sans changer les noms, les signatures ni ce que rendent les fonctions.

Ce qu'il devrait faire, comme ton `BPETokenizer` de B2.17 :
- `train_18(text, vocab_size)` rend `(merges, words)` : les fusions dans l'ordre d'apprentissage, et chaque pré-token du texte avec ses identifiants à la fin de l'entraînement ;
- `encode_18(text, merges)` encode par rang ;
- `decode_18(ids, merges)` redonne le texte exact, et un octet seul devient « � ».

La cellule de vérification lance `diagnose_18` : un entraînement de 300 tokens sur les 120 000 premiers caractères de `VERNE_TRAIN` (environ 1 s), puis trois diagnostics enchaînés, chacun ne s'exécutant que si le précédent passe, car une erreur peut en cacher une autre :
- `decode` redonne-t-il un texte d'essai ?
- comment « é » est-il encodé ?
- `encode` découpe-t-il les mots du texte d'entraînement comme l'entraînement les a laissés ?

Elle tourne dès le premier « Tout exécuter » : ses messages sont ton point de départ. Une fois les trois points passés, la cellule suivante mesure ta version réparée sur la validation de Verne.

Dans tes notes : les trois erreurs, et pour chacune le symptôme qu'elle produit seule. Laquelle aurait laissé passer un test d'aller-retour sur un texte anglais sans accent ni emoji ? Laquelle ne change jamais le texte rendu par `decode`, seulement le nombre de tokens ?

In [36]:
TRAIN_18 = VERNE_TRAIN[:120_000]          # the training text of the diagnostics (≈ 1 s per training)
PROBE_18 = "« Phileas Fogg a gagné son pari ! » 🙂"


def diagnose_18(train, encode, decode, vocab_size=300):
    """Three diagnoses of a small tokenizer, in a row: the first one that fails stops the chain (one error can hide
    another). Returns True when the three pass."""
    result = train(TRAIN_18, vocab_size)
    if not (isinstance(result, tuple) and len(result) == 2 and isinstance(result[1], dict)):
        print(f"❌ Ex B2.18 : train_18 doit rendre le couple (merges, words) ; reçu {type(result).__name__}.")
        return False
    merges, words = result
    vocab = {i: bytes([i]) for i in range(256)}
    for r, (a, b) in enumerate(merges):
        vocab[256 + r] = vocab[a] + vocab[b]

    def shown(ids):
        return "[" + ", ".join(repr(vocab[i].decode("utf-8", errors="replace")) if i in vocab else str(i) for i in ids) + "]"

    def point1():
        try:
            back = decode(encode(PROBE_18, merges), merges)
        except UnicodeDecodeError as exc:
            return False, f"decode(encode(texte)) lève UnicodeDecodeError ({exc.reason}) sur « {PROBE_18} »", \
                "ce texte contient des caractères de plusieurs octets : à quel moment decode transforme-t-il des octets en texte ?"
        if back != PROBE_18:
            same_screen = unicodedata.normalize("NFC", back) == PROBE_18
            return False, f"decode(encode(texte)) rend {back!r} au lieu de {PROBE_18!r}", \
                ("les deux textes s'affichent pareil, mais leurs points de code diffèrent : compare-les"
                 if same_screen else "quels caractères changent, et combien d'octets ont-ils ?")
        try:
            lone = decode([0xC3], merges)
        except UnicodeDecodeError as exc:
            return False, f"decode([0xC3]) (le premier octet de « é », seul) lève UnicodeDecodeError ({exc.reason})", \
                "relis la docstring de decode_18 : que doit devenir un octet seul ?"
        if lone != "�":
            return False, f"decode([0xC3]) rend {lone!r} au lieu de '�'", "relis la docstring de decode_18."
        return True, "le texte d'essai revient à l'identique, et un octet seul donne '�'", ""

    def point2():
        ids = encode("é", merges)
        n = TRAIN_18.count("é")
        if len(ids) == 1:
            return True, f"« é » ({n} fois dans le texte d'entraînement) est un seul token", ""
        return False, f"« é » apparaît {n} fois dans le texte d'entraînement, mais encode le découpe en {len(ids)} tokens {[vocab.get(i, i) for i in ids]}", \
            "les fusions apprises pour « é » portent-elles sur les octets que reçoit encode ?"

    def point3():
        counts = Counter(PIECES_18.findall(TRAIN_18))
        diff = sorted((w for w in words if encode(w, merges) != words[w]), key=lambda w: (-counts.get(w, 0), w))
        if not diff:
            return True, f"les {len(words)} mots distincts du texte d'entraînement sont découpés par encode comme à la fin de l'entraînement", ""
        w = diff[0]
        return False, (f"{len(diff)} des {len(words)} mots distincts du texte d'entraînement ne sont pas découpés par encode "
                       f"comme à la fin de l'entraînement ; par exemple {w!r} : {shown(words[w])} à l'entraînement, "
                       f"{shown(encode(w, merges))} par encode"), \
            "compare les deux découpages : laquelle de ses fusions encode a-t-il faite en premier ?"

    points = [("decode redonne le texte", point1), ("« é » est un token", point2),
              ("encode découpe comme l'entraînement", point3)]
    for k, (name, run) in enumerate(points, start=1):
        ok, measured, why = run()
        if ok:
            print(f"✅ Ex B2.18 : point {k}, {name} ({measured}).")
        else:
            print(f"❌ Ex B2.18 : point {k} : {measured} ; {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex B2.18 : point {later} : corrige d'abord le point {k}.")
            return False
    print("✅ Ex B2.18 : les trois diagnostics passent.")
    return True

In [37]:
# The tokenizer to repair IN PLACE (same names, same signatures, same returns): the diagnostics of the next cell start
# from it
PIECES_18 = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+")   # GPT-2 pre-tokens


def clean_18(text):
    """The text as the tokenizer learns it: accents written in a single standard form."""
    return unicodedata.normalize("NFD", text)


def merge_18(ids, pair, new_id):
    """A new list where each occurrence of pair, left to right and without overlap, becomes new_id."""
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def train_18(text, vocab_size):
    """Byte-level BPE learned on text (GPT-2 pre-tokens; ties go to the smallest pair). Returns (merges, words): the
    merges in learning order, and {pre-token: its ids at the end of the training}."""
    text = clean_18(text)
    counts = Counter(PIECES_18.findall(text))
    words = {w: list(w.encode("utf-8")) for w in counts}
    merges = []
    while 256 + len(merges) < vocab_size:
        pairs = Counter()
        for w, ids in words.items():
            for pair in zip(ids, ids[1:]):
                pairs[pair] += counts[w]
        if not pairs:
            break
        best = min(pairs, key=lambda p: (-pairs[p], p))
        merges.append(best)
        words = {w: merge_18(ids, best, 255 + len(merges)) for w, ids in words.items()}
    return merges, words


def encode_18(text, merges):
    """The ids of text: inside each pre-token, its UTF-8 bytes, then the learned merges until none applies."""
    rank = {pair: r for r, pair in enumerate(merges)}
    out = []
    for piece in PIECES_18.findall(text):
        ids = list(piece.encode("utf-8"))
        i = 0
        while i < len(ids) - 1:
            pair = (ids[i], ids[i + 1])
            if pair in rank:
                ids[i:i + 2] = [256 + rank[pair]]
                i = 0
            else:
                i += 1
        out += ids
    return out


def decode_18(ids, merges):
    """The text of ids; a byte left alone (part of a character) becomes '�'."""
    vocab = {i: bytes([i]) for i in range(256)}
    for r, (a, b) in enumerate(merges):
        vocab[256 + r] = vocab[a] + vocab[b]
    return "".join(vocab[i].decode("utf-8") for i in ids)

In [38]:
with checked("B2.18"):
    ok_18 = diagnose_18(train_18, encode_18, decode_18)

❌ Ex B2.18 : point 1 : decode(encode(texte)) lève UnicodeDecodeError (unexpected end of data) sur « « Phileas Fogg a gagné son pari ! » 🙂 » ; ce texte contient des caractères de plusieurs octets : à quel moment decode transforme-t-il des octets en texte ?
⏳ Ex B2.18 : point 2 : corrige d'abord le point 1.
⏳ Ex B2.18 : point 3 : corrige d'abord le point 1.


In [39]:
if not globals().get("ok_18"):
    print("⏳ Ex B2.18 : la comparaison attend que les trois points de la cellule précédente passent.")
else:
    with checked("B2.18"):
        merges_18, _ = train_18(TRAIN_18, 300)
        val_18 = VERNE_VAL[:20_000]
        ids_18 = encode_18(val_18, merges_18)
        print(f"Ta version réparée : {fr(n_bytes(val_18) / len(ids_18), 2)} octets par token sur 20 000 caractères de la "
              f"validation de Verne ; « {PROBE_18} » →", [decode_18([i], merges_18) for i in encode_18(PROBE_18, merges_18)])

⏳ Ex B2.18 : la comparaison attend que les trois points de la cellule précédente passent.


### Ex B2.19 — Ton BPE contre la bibliothèque tokenizers : mêmes fusions, autre vitesse 📦 ★★ ⏱️ 25 min
**Objectif :** entraîner un BPE avec la bibliothèque `tokenizers`, lire ses fusions dans sa description JSON, et vérifier que ton BPE apprend exactement les mêmes.  
**Prérequis :** Ex B2.17 · fiche §202.3 (encadré 🕰️ sur le départage), §202.5 (« Les tokenizers des modèles, lus dans leur `tokenizer.json` », « Entraîner son propre tokenizer avec `tokenizers` ») · **Fil rouge :** Holmes/Verne · **Parcours :** M, C

**Contrat**
- `train_hf_19(text, vocab_size, min_frequency=2)` : un `Tokenizer` de `tokenizers`, de modèle `models.BPE()` et de pré-tokenizer `pre_tokenizers.Whitespace()`, entraîné sur `text` par un `BpeTrainer` de ces `vocab_size` et `min_frequency`, sans token spécial ni barre de progression ;
- `hf_merges_19(hf_tok)` : ses fusions, en liste de tuples `(id gauche, id droite)` d'`int`, dans l'ordre d'apprentissage. Lis-les dans sa description JSON, `hf_tok.to_str()`, qui est le contenu de son `tokenizer.json`, avec son vocabulaire `hf_tok.get_vocab()`.

La cellule de vérification entraîne ta fonction sur `VERNE_TRAIN` avec 2 000 tokens. Elle vérifie trois valeurs, calculées par tes fonctions (rien à taper) :
- a) la taille de l'alphabet de base, la taille du vocabulaire moins le nombre de fusions ;
- b) le rang de la fusion qui crée le token « Passepartout » ;
- c) le nombre de tokens de `VERNE_VAL`.

Elle entraîne ensuite **ton** `BPETokenizer` de la même façon (niveau caractère, `'whitespace'`, 2 000 tokens), compare ses fusions et ses identifiants à ceux de `tokenizers`, et affiche les deux temps d'entraînement. Ce temps est une mesure, pas une réponse : il dépend de la machine, et de ta façon de compter les paires (de 1 s à deux minutes environ).

Dans tes notes : combien de fois ton entraînement est-il plus lent que celui de `tokenizers`, et d'où vient l'écart (le langage, l'algorithme) ? Que dit le rang de « Passepartout » de sa fréquence dans le roman ? Pourquoi la comparaison exacte n'est-elle possible qu'avec la même règle de départage et le même ordre de l'alphabet ?

In [40]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers


def train_hf_19(text, vocab_size, min_frequency=2):
    """A Tokenizer of the library tokenizers: models.BPE, pre_tokenizers.Whitespace, trained by BpeTrainer on text
    (no special token, no progress bar)."""
    raise NotImplementedError("train_hf_19")


def hf_merges_19(hf_tok):
    """The merges of a trained BPE of tokenizers, as (left_id, right_id) tuples of ints, in learning order (rank)."""
    raise NotImplementedError("hf_merges_19")

In [41]:
VOCAB_19 = 2000
ok_hf_19 = False
with checked("B2.19"):
    start_19 = time.perf_counter()
    hf_19 = train_hf_19(VERNE_TRAIN, VOCAB_19)
    seconds_hf_19 = time.perf_counter() - start_19
    merges_hf_19 = hf_merges_19(hf_19)
    if not returned("B2.19", "hf_merges_19", merges_hf_19):
        raise StopCheck
    if not (isinstance(merges_hf_19, list) and all(type(m) is tuple and len(m) == 2 and all(isinstance(i, int) for i in m)
                                                   for m in merges_hf_19)):
        print("❌ Ex B2.19 : hf_merges_19 doit rendre une liste de tuples de deux int (les ids des deux tokens fusionnés).")
        raise StopCheck
    n_base_19 = hf_19.get_vocab_size() - len(merges_hf_19)
    strings_19 = {i: s for s, i in hf_19.get_vocab().items()}
    rank_19 = next((r for r, (a, b) in enumerate(merges_hf_19) if strings_19.get(a, "") + strings_19.get(b, "") == "Passepartout"), None)
    val_ids_19 = hf_19.encode(VERNE_VAL).ids
    print(f"tokenizers : {hf_19.get_vocab_size()} tokens, dont {len(merges_hf_19)} fusions, appris en {fr(seconds_hf_19, 2)} s ; "
          f"la validation de Verne fait {len(val_ids_19)} tokens.")
    wb.check("B2.19a", n_base_19, computed=True)
    wb.check("B2.19b", rank_19, computed=True)
    wb.check("B2.19c", len(val_ids_19), computed=True)
    ok_hf_19 = True

if not bpe_written():
    print("⏳ Ex B2.19 : la comparaison à ton BPE attend BPETokenizer.fit et encode (B2.16 et B2.17).")
elif not ok_hf_19:
    print("⏳ Ex B2.19 : la comparaison à ton BPE attend train_hf_19 et hf_merges_19 (cellule précédente).")
else:
    with checked("B2.19"):
        start_19 = time.perf_counter()
        mine_19 = mylearn.tokenizer.BPETokenizer(vocab_size=VOCAB_19, level="char", pre_tokenizer="whitespace").fit(VERNE_TRAIN)
        seconds_mine_19 = time.perf_counter() - start_19
        first_19 = next((r for r, (a, b) in enumerate(zip(mine_19.merges_, merges_hf_19)) if tuple(a) != tuple(b)), None)
        verdict("B2.19", list(map(tuple, mine_19.merges_)) == merges_hf_19,
                f"ton BPE et tokenizers apprennent les mêmes {len(merges_hf_19)} fusions, dans le même ordre.",
                (f"ta fusion de rang {first_19} est {mine_19.merges_[first_19]}, celle de tokenizers {merges_hf_19[first_19]}"
                 if first_19 is not None else f"{len(mine_19.merges_)} fusions chez toi, {len(merges_hf_19)} chez tokenizers")
                + " : relis le départage et l'alphabet (B2.16), puis hf_merges_19.")
        verdict("B2.19", mine_19.encode(VERNE_VAL) == val_ids_19,
                "et les mêmes ids pour toute la validation de Verne.",
                "les ids de la validation diffèrent de ceux de tokenizers : relis encode (B2.17).")
        print(f"Temps de l'entraînement : ton BPETokenizer {fr(seconds_mine_19, 2)} s, tokenizers {fr(seconds_hf_19, 2)} s "
              f"(rapport {seconds_mine_19 / seconds_hf_19:.0f}, sur cette machine, pour cette exécution).")

⏳ Ex B2.19 : pas encore fait (train_hf_19).
⏳ Ex B2.19 : la comparaison à ton BPE attend BPETokenizer.fit et encode (B2.16 et B2.17).


## Partie C · Expériences : la taille du vocabulaire et la langue d'entraînement

Deux expériences avec ton `BPETokenizer` au niveau octet et le pré-tokenizer `'gpt2'` (sans perte). D'abord la taille du vocabulaire, de 300 à 8 000 tokens (B2.20) ; puis ce que coûte un tokenizer appris sur une langue quand on l'applique à l'autre (B2.21). Les mesures se font toujours sur la **validation**, que les tokenizers n'ont jamais vue.

Deux outils (dans la cellule des outils) :
- `fast_byte_bpe(texts, vocab_size)` rend un `BPETokenizer` de **ta** librairie dont les fusions sont apprises par `tokenizers`, en Rust : exactement celles de ton `fit` (B2.19 l'a montré au niveau caractère, B2.20 le vérifie au niveau octet), en moins d'une seconde pour tout Verne, là où une version Python qui recompte toutes les paires à chaque fusion met une à quatre minutes. Ton `encode` fait tout le reste ;
- `truncated_bpe(tok, taille)` garde les `taille − 256` premières fusions d'un tokenizer : les fusions s'apprennent une à une, et les premières sont les mêmes quelle que soit la taille finale.

### Ex B2.20 — Taille du vocabulaire : compression et entropie par token, de 300 à 8 000 tokens 🔬 ★★★ ⏱️ 40 min
**Objectif :** mesurer ce qu'apporte un vocabulaire plus grand, en compression et en information par token, pour choisir une taille.  
**Prérequis :** Ex B2.19 · ch. 6 (entropie, code de Huffman) · fiche §202.3, §202.6 · **Fil rouge :** Holmes/Verne · **Parcours :** M, C

Quelle taille de vocabulaire choisir ? Plus de tokens raccourcissent les textes, mais agrandissent la table d'embedding et rendent chaque token plus rare. Tu mesures sur `VERNE_VAL`, pour six tailles de 300 à 8 000 :
- la **compression**, en octets par token ;
- l'**entropie** de la distribution des tokens, en bits par token : $H = -\sum_t p_t \log_2 p_t$, où $p_t$ est la fréquence de l'identifiant $t$ dans le texte encodé (ch. 6) ;
- les **bits par octet**, $H$ divisé par la compression : ce que coûterait chaque octet du texte avec un code de Huffman idéal sur les tokens, si on les supposait indépendants (ch. 6).

**Contrat**
- `compression_20(tok, text)` : le nombre d'octets UTF-8 de `text` divisé par le nombre d'identifiants de `tok.encode(text)` ;
- `token_entropy_20(tok, text)` : l'entropie en bits (logarithme de base 2) des fréquences des identifiants de `tok.encode(text)` ;
- `sweep_20(big_tok, text, sizes)` : une ligne par taille de `sizes`, dans leur ordre ; chaque ligne est un dictionnaire de clés `"vocab_size"`, `"bytes_per_token"`, `"bits_per_token"`, `"bits_per_byte"` et `"used"` (le nombre d'identifiants distincts du texte encodé), mesurées sur `text` avec `truncated_bpe(big_tok, size)`.

La cellule de vérification essaie tes deux mesures sur un faux tokenizer, qui rend des identifiants fixés à l'avance. La cellule d'expérience apprend un BPE de 8 000 tokens sur `VERNE_TRAIN` avec `fast_byte_bpe`. Elle vérifie à 400 tokens que ton `fit` apprend les mêmes fusions (sur 100 000 caractères, de 1 à 10 s selon ta façon de compter les paires) et que `truncated_bpe` redonne bien le tokenizer plus petit. Elle lance enfin `sweep_20` et trace les trois courbes.

Dans tes notes :
- la compression gagne-t-elle autant à chaque doublement de la taille ?
- l'entropie par token et les bits par octet : lequel monte, lequel baisse, et comment les deux vont-ils ensemble ?
- combien des 8 000 tokens servent dans la validation (colonne `used`) ?
- quelle taille prendrais-tu pour un petit modèle de langue sur Verne, et que ces courbes ne disent-elles pas (la taille de la table d'embedding, 🧮 B2.9 ; les tokens rares, vus trop peu pour être bien appris) ?

In [42]:
def compression_20(tok, text):
    """The bytes per token of text: its number of UTF-8 bytes / the number of ids of tok.encode(text)."""
    raise NotImplementedError("compression_20")


def token_entropy_20(tok, text):
    """The entropy, in bits, of the frequencies of the ids of tok.encode(text)."""
    raise NotImplementedError("token_entropy_20")


def sweep_20(big_tok, text, sizes):
    """One dict per size of sizes, in their order: {"vocab_size", "bytes_per_token", "bits_per_token",
    "bits_per_byte", "used"} measured on text with truncated_bpe(big_tok, size); bits_per_byte = bits_per_token /
    bytes_per_token, "used" = the number of distinct ids of the encoded text."""
    raise NotImplementedError("sweep_20")

In [43]:
class ListTokenizer_20:
    """A fake tokenizer for the check: encode gives the ids it was built with, whatever the text."""
    def __init__(self, ids):
        self.ids = list(ids)

    def encode(self, text):
        return list(self.ids)


ok_20 = False
with checked("B2.20"):
    c_20 = compression_20(ListTokenizer_20([7, 8]), "été")
    if returned("B2.20", "compression_20", c_20):
        verdict("B2.20", isinstance(c_20, (int, float)) and math.isclose(c_20, 2.5),
                "compression_20 : « été » en 2 tokens, 5 octets UTF-8, soit 2,5 octets par token.",
                f"« été » (5 octets UTF-8, 3 caractères) encodé en 2 tokens : attendu 2,5 octets par token ; reçu {c_20!r}.")
    h_20 = token_entropy_20(ListTokenizer_20([4, 4, 9, 2]), "x")
    if returned("B2.20", "token_entropy_20", h_20):
        verdict("B2.20", isinstance(h_20, (int, float)) and math.isclose(h_20, 1.5),
                "token_entropy_20 : les ids [4, 4, 9, 2] ont une entropie de 1,5 bit.",
                f"les ids [4, 4, 9, 2] (fréquences 1/2, 1/4, 1/4) : attendu 1,5 bit (logarithme de base 2) ; reçu {h_20!r}.")
    h0_20 = token_entropy_20(ListTokenizer_20([3, 3, 3]), "x")
    verdict("B2.20", isinstance(h0_20, (int, float)) and math.isclose(h0_20, 0.0, abs_tol=1e-12),
            "un seul id répété : entropie nulle.", f"un seul id répété doit donner une entropie nulle ; reçu {h0_20!r}.")
    ok_20 = (isinstance(c_20, (int, float)) and math.isclose(c_20, 2.5) and isinstance(h_20, (int, float))
             and math.isclose(h_20, 1.5) and math.isclose(h0_20, 0.0, abs_tol=1e-12))

⏳ Ex B2.20 : pas encore fait (compression_20).


In [44]:
SIZES_20 = (300, 500, 1000, 2000, 4000, 8000)
if not bpe_written():
    print("⏳ Ex B2.20 : l'expérience attend BPETokenizer.fit et encode (B2.16 et B2.17).")
elif not globals().get("ok_20"):
    print("⏳ Ex B2.20 : l'expérience attend que compression_20 et token_entropy_20 passent la cellule précédente.")
else:
    with checked("B2.20"):
        start_20 = time.perf_counter()
        BIG_20 = fast_byte_bpe(VERNE_TRAIN, max(SIZES_20))
        print(f"Les {len(BIG_20.merges_)} fusions d'un BPE octet de {max(SIZES_20)} tokens sur VERNE_TRAIN, apprises par "
              f"tokenizers : {fr(time.perf_counter() - start_20, 1)} s.")
        part_20 = VERNE_TRAIN[:100_000]
        start_20 = time.perf_counter()
        mine_20 = mylearn.tokenizer.BPETokenizer(vocab_size=400, pre_tokenizer="gpt2").fit(part_20)
        seconds_20 = time.perf_counter() - start_20
        verdict("B2.20", mine_20.merges_ == fast_byte_bpe(part_20, 400).merges_,
                f"ton fit (400 tokens sur 100 000 caractères de Verne, {fr(seconds_20, 1)} s) apprend les mêmes fusions que "
                "fast_byte_bpe.",
                "ton fit et fast_byte_bpe n'apprennent pas les mêmes fusions sur 100 000 caractères de Verne : relis "
                "B2.16 (départage, min_frequency, niveau octet).")
        verdict("B2.20", truncated_bpe(BIG_20, 400).merges_ == fast_byte_bpe(VERNE_TRAIN, 400).merges_,
                f"les 144 premières fusions de l'entraînement à {max(SIZES_20)} tokens sont celles d'un entraînement à 400 : "
                "truncated_bpe redonne le tokenizer plus petit.",
                "les premières fusions d'un entraînement long devraient être celles d'un entraînement court.")
        rows_20 = sweep_20(BIG_20, VERNE_VAL, SIZES_20)
        if not returned("B2.20", "sweep_20", rows_20):
            raise StopCheck
        table_20 = pd.DataFrame(rows_20)
        need_20 = ["vocab_size", "bytes_per_token", "bits_per_token", "bits_per_byte", "used"]
        if list(table_20.get("vocab_size", [])) != list(SIZES_20) or any(c not in table_20 for c in need_20):
            print(f"❌ Ex B2.20 : sweep_20 doit rendre une ligne par taille de {SIZES_20}, dans cet ordre, avec les clés {need_20}.")
            raise StopCheck
        row_20 = rows_20[SIZES_20.index(1000)]
        tok1000_20 = truncated_bpe(BIG_20, 1000)
        ids1000_20 = tok1000_20.encode(VERNE_VAL)
        verdict("B2.20", math.isclose(row_20["bytes_per_token"], compression_20(tok1000_20, VERNE_VAL))
                and math.isclose(row_20["bits_per_token"], token_entropy_20(tok1000_20, VERNE_VAL))
                and math.isclose(row_20["bits_per_byte"], row_20["bits_per_token"] / row_20["bytes_per_token"])
                and row_20["used"] == len(set(ids1000_20)),
                "à 1 000 tokens, la ligne de sweep_20 reprend tes deux mesures, leur quotient et le nombre d'ids distincts.",
                "à 1 000 tokens, la ligne de sweep_20 doit reprendre compression_20, token_entropy_20 (sur "
                "truncated_bpe(big_tok, 1000)), leur quotient et le nombre d'identifiants distincts du texte encodé.")
        print(table_20[need_20].to_string(index=False, float_format=lambda x: f"{x:.3f}"))
        fig_20, axes_20 = plt.subplots(1, 3, figsize=(13, 3.6))
        for ax, col, label in zip(axes_20, ("bytes_per_token", "bits_per_token", "bits_per_byte"),
                                  ("octets par token", "entropie (bits par token)", "bits par octet")):
            ax.plot(table_20["vocab_size"], table_20[col], "o-")
            ax.set_xscale("log")
            ax.set_xticks(SIZES_20, [str(s) for s in SIZES_20])
            ax.set_xlabel("taille du vocabulaire")
            ax.set_title(label)
            ax.grid(alpha=0.3)
        fig_20.suptitle("BPE octet de Verne (VERNE_TRAIN), mesuré sur VERNE_VAL")
        fig_20.tight_layout()
        plt.show()

⏳ Ex B2.20 : l'expérience attend BPETokenizer.fit et encode (B2.16 et B2.17).


### Ex B2.21 — Tokenizer entraîné sur Holmes, appliqué à Verne (et l'inverse) : le coût de la langue 🔬 ★★ ⏱️ 25 min
**Objectif :** mesurer ce que coûte un tokenizer appris sur une autre langue, et ce que coûte un tokenizer partagé.  
**Prérequis :** Ex B2.20 · Ex B2.13 · fiche §202.6 (« Le même texte, un autre prix selon la langue ») · **Fil rouge :** Holmes/Verne · **Parcours :** C

En B2.13, GPT-2 compresse moins bien Verne que Holmes. Cela vient-il du français, ou des textes sur lesquels son tokenizer a appris ? Tu entraînes trois BPE octet de 2 000 tokens et tu les mesures sur la validation des deux romans.

**Contrat**
- `train_tokenizers_21(vocab_size=2000)` : trois `BPETokenizer` de ta librairie, au niveau octet, de pré-tokenizer `'gpt2'` et de `vocab_size` tokens, appris par `fast_byte_bpe` (ou par ton `fit`, plus lent, qui donne les mêmes fusions), dans un dictionnaire :
  - `"holmes"`, appris sur `HOLMES_TRAIN` ;
  - `"verne"`, appris sur `VERNE_TRAIN` ;
  - `"mixte"`, appris sur les deux textes ensemble.
- `cross_table_21(tokenizers, texts)` : un `DataFrame` de pandas des octets par token (`compression_20`, B2.20), une ligne par tokenizer et une colonne par texte ; `tokenizers` et `texts` sont des dictionnaires nom → objet.

La cellule de vérification entraîne tes trois tokenizers (quelques secondes avec `fast_byte_bpe`) et contrôle leurs réglages. Elle vérifie aussi que celui de Verne commence par les fusions de B2.20. Elle calcule ensuite ton tableau, y ajoute GPT-2 et le trace.

Dans tes notes :
- de combien la compression baisse-t-elle quand on applique un tokenizer à l'autre roman ?
- le tokenizer mixte : que coûte-t-il à chaque langue, par rapport à son tokenizer « maison » ?
- où se place GPT-2, avec ses 50 257 tokens ?
- qu'en conclus-tu sur la « taxe » de certaines langues (fiche §202.6), et sur le choix des données d'entraînement du tokenizer d'un modèle multilingue ?

In [45]:
def train_tokenizers_21(vocab_size=2000):
    """Three byte-level BPETokenizer of your library, pre_tokenizer "gpt2", vocab_size tokens:
    {"holmes": learned on HOLMES_TRAIN, "verne": on VERNE_TRAIN, "mixte": on both texts}."""
    raise NotImplementedError("train_tokenizers_21")


def cross_table_21(tokenizers, texts):
    """A pandas DataFrame of the bytes per token (compression_20): one row per tokenizer, one column per text
    (tokenizers and texts are dicts name -> object)."""
    raise NotImplementedError("cross_table_21")

In [46]:
if not bpe_written():
    print("⏳ Ex B2.21 : l'expérience attend BPETokenizer.fit et encode (B2.16 et B2.17).")
else:
    with checked("B2.21"):
        need("B2.21", "compression_20")
        start_21 = time.perf_counter()
        toks_21 = train_tokenizers_21(2000)
        print(f"Trois tokenizers de 2 000 tokens : {fr(time.perf_counter() - start_21, 1)} s.")
        if not returned("B2.21", "train_tokenizers_21", toks_21):
            raise StopCheck
        if not isinstance(toks_21, dict) or set(toks_21) != {"holmes", "verne", "mixte"}:
            print("❌ Ex B2.21 : train_tokenizers_21 doit rendre un dictionnaire de clés 'holmes', 'verne' et 'mixte'.")
            raise StopCheck
        for name, tok in toks_21.items():
            ok = (getattr(tok, "level", None) == "byte" and getattr(tok, "pre_tokenizer", None) == "gpt2"
                  and len(getattr(tok, "vocab_", {})) == 2000 and not getattr(tok, "special_ids_", {}))
            verdict("B2.21", ok, f"{name} : BPE octet 'gpt2' de 2 000 tokens.",
                    f"{name} : attendu un BPETokenizer de niveau octet, pré-tokenizer 'gpt2', 2 000 tokens sans token spécial.")
        if "BIG_20" in globals():
            verdict("B2.21", toks_21["verne"].merges_ == BIG_20.merges_[:2000 - 256],
                    "le tokenizer de Verne a les 1 744 premières fusions de celui de B2.20 : il est appris sur VERNE_TRAIN.",
                    "le tokenizer de Verne devrait avoir les premières fusions de BIG_20 (B2.20) : sur quel texte est-il appris ?")
        verdict("B2.21", toks_21["holmes"].merges_[:20] != toks_21["verne"].merges_[:20]
                and toks_21["mixte"].merges_ not in (toks_21["holmes"].merges_, toks_21["verne"].merges_),
                "les trois tokenizers ont des fusions différentes.",
                "deux de tes tokenizers ont les mêmes fusions : sont-ils appris sur des textes différents ?")
        texts_21 = {"Holmes (validation)": HOLMES_VAL, "Verne (validation)": VERNE_VAL}
        table_21 = cross_table_21(toks_21, texts_21)
        if not returned("B2.21", "cross_table_21", table_21):
            raise StopCheck
        if not isinstance(table_21, pd.DataFrame) or set(table_21.index) != set(toks_21) or set(table_21.columns) != set(texts_21):
            print("❌ Ex B2.21 : cross_table_21 doit rendre un DataFrame, une ligne par tokenizer, une colonne par texte.")
            raise StopCheck
        verdict("B2.21", math.isclose(float(table_21.loc["verne", "Verne (validation)"]),
                                      compression_20(toks_21["verne"], VERNE_VAL)),
                "le tableau contient les octets par token de compression_20.",
                "le tableau doit contenir compression_20(tokenizer, texte) : relis-le.")
        gpt2_21 = wbtext.try_tokenizer("gpt2", "B2.21")
        if gpt2_21 is not None:
            table_21.loc["GPT-2 (50 257 tokens)"] = pd.Series({name: n_bytes(t) / wbtext.count_tokens(gpt2_21, t)
                                                              for name, t in texts_21.items()})
        print(table_21.to_string(float_format=lambda x: f"{x:.2f}"))
        ax_21 = table_21.plot.bar(figsize=(7.5, 3.6), rot=0)
        ax_21.set_ylabel("octets par token")
        ax_21.set_title("Chaque tokenizer sur la validation des deux textes")
        ax_21.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()

⏳ Ex B2.21 : l'expérience attend BPETokenizer.fit et encode (B2.16 et B2.17).


## Partie D · Des vecteurs de mots : compter, factoriser, prédire

Deuxième phase : des tokens aux vecteurs. Tu écris dans `mylearn/embeddings.py` la matrice de co-occurrences et la PPMI (B2.22), puis le cosinus et les plus proches voisins (B2.23). Tu construis des vecteurs de mots de Holmes par PPMI + SVD (B2.24), puis tu entraînes un word2vec en PyTorch sur Verne (B2.25), dont tu prédis les voisins de deux personnages (B2.26) avant de lire ses cartes (B2.27). `cosine_similarity_matrix` et `most_similar` resserviront en B4 (recherche dense).

La cellule suivante importe PyTorch et prépare les mots des deux romans, comme la carte de la fiche (§202.9) : `WORDS_H`, en minuscules et en lettres `a` à `z` ; `WORDS_V`, en minuscules, accents gardés (« été », « où »). Elle donne aussi `nearest_words(E, itos, mot, k)`, les $k$ plus proches voisins d'un mot dans une matrice de vecteurs de norme 1, pour les cellules de vérification. Au ch. 22, tu avais déjà découpé ces deux textes en mots pour en faire un vocabulaire ; ici, chaque mot devient un vecteur.

**Contrat commun de `embeddings.py`** (les docstrings, plus ces précisions que vérifient les tests) :
- tes fonctions acceptent des listes et des tableaux d'entiers ou de flottants, et ne modifient jamais leurs entrées ;
- une entrée invalide lève `ValueError`, jamais `IndexError` ni un résultat silencieusement faux ;
- NumPy seulement ; les tests comparent à une double boucle, à la formule en `float64`, à scikit-learn et à PyTorch, à `rtol = 1e-6` et `atol = 1e-6` près (un calcul en `float32` passe ; la tolérance s'affiche dans le message).

In [47]:
# Tools of the phase « embeddings » (parts D and E)
from wb.nbtools import pair

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:                       # a computer without PyTorch (Mac Intel): parts D and E are done on Colab
    torch = None
    print("⏳ PyTorch absent sur cet ordinateur : B2.25 à B2.29 se font sur Colab (00_setup/INSTALL_LOCAL.md) ; les "
          "tests de ta librairie tournent sans lui : python -m pytest tests/test_chb2_embeddings.py -q")

    def _skip_torch_cells(lines):
        """IPython input transformer: without PyTorch, a cell that uses it prints ⏳ instead of stopping "Run all"."""
        if re.search(r"\b(torch|torchvision)\b|(?<![\w.])(nn|F)\.", "".join(lines)):
            return ['print("⏳ PyTorch absent sur cet ordinateur : cette cellule se fait sur Colab.")\n']
        return lines

    try:
        _shell = get_ipython()
    except NameError:                     # not in a notebook
        _shell = None
    if _shell is not None and all(getattr(f, "__name__", "") != "_skip_torch_cells" for f in _shell.input_transformers_cleanup):
        _shell.input_transformers_cleanup.append(_skip_torch_cells)

TEST_EMB = "tests/test_chb2_embeddings.py"     # the tests of your library embeddings.py
WORDS_H = wbtext.words("holmes")               # the 105 863 words of Holmes, [a-z]+ in lower case (the map of the fiche)
WORDS_V = wbtext.words("verne")                # the 72 367 words of Verne, accents kept


def nearest_words(E, itos, word, k=10, stoi=None):
    """The k nearest other words of `word` by cosine (rows of E of norm 1), ties to the smaller id: [(word, cos)]."""
    stoi = stoi or {w: i for i, w in enumerate(itos)}
    sims = E @ E[stoi[word]]
    order = [i for i in np.argsort(-sims, kind="stable") if i != stoi[word]][:k]
    return [(itos[i], float(sims[i])) for i in order]


print(f"Holmes : {len(WORDS_H):_} mots ({len(set(WORDS_H)):_} formes) ; Verne : {len(WORDS_V):_} mots "
      f"({len(set(WORDS_V)):_} formes).".replace("_", " "))

Holmes : 105 863 mots (7 817 formes) ; Verne : 72 367 mots (8 793 formes).


### Ex B2.22 — cooccurrence_matrix et ppmi 🔨 ★★ ⏱️ 30 min
**Objectif :** compter les co-occurrences d'une suite de mots dans une fenêtre, puis les transformer en information mutuelle ponctuelle positive.  
**Prérequis :** ✏️ B2.6 · ch. 6 (information, divergence KL) · fiche §202.8 (« La matrice de co-occurrences », « PMI et PPMI ») · **Fil rouge :** Holmes · **mylearn :** `embeddings.py` · **Parcours :** M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/embeddings.py` (créé par `python tools/start_chapter.py B2`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb2_embeddings.py -q` les lance tous dans un terminal.

**Contrat**
- `cooccurrence_matrix(ids, vocab_size, window=2, weighting="uniform", symmetric=True)` :
  - rend une matrice **float** `(vocab_size, vocab_size)`, même en `'uniform'` ;
  - pour chaque position `i` et chaque distance `d` de 1 à `window`, ajoute le poids (1, ou `1 / d` en `'harmonic'`) à `C[ids[i], ids[i + d]]` et, si `symmetric`, à `C[ids[i], ids[i - d]]` ; avec `symmetric=False`, seulement le contexte de droite ;
  - les répétitions d'une même paire s'**additionnent** : `[0, 0, 0, 1]` avec une fenêtre de 1 donne `C[0, 0] = 4` ;
  - les ids sont des entiers (une liste, des `int32` ou des `uint8` conviennent) ; une suite plus courte que la fenêtre est permise ;
  - `ValueError` pour des ids qui ne sont pas 1-D, `window < 1`, une pondération inconnue, et un id hors de `[0, vocab_size)`, −1 compris (pas d'`IndexError`, pas d'indice négatif à la Python).
- `ppmi(C, alpha=1.0)` :
  - $\max\big(0, \log \frac{p(w, c)}{p(w)\,p_\alpha(c)}\big)$ avec le **logarithme naturel**, les lignes pour les mots, les colonnes pour les contextes ;
  - le lissage `alpha` ne porte que sur les **colonnes** : $p_\alpha(c) = \#(c)^\alpha / \sum_{c'} \#(c')^\alpha$ (renormalisé) ;
  - 0 là où `C == 0`, donc une ligne ou une colonne nulle donne des 0, **jamais NaN** ;
  - accepte des entiers et des listes ; `ValueError` pour `C` qui n'est pas 2-D, une entrée négative ou une somme nulle.

La première cellule de vérification essaie tes deux fonctions sur des cas visibles (la matrice chien / chat / os de la fiche, un mot jamais vu, une matrice « indépendante »), compare tes comptes sur 30 000 mots de Holmes à ceux de `scipy.sparse`, puis essaie huit refus. La seconde lance les 36 tests de l'exercice (`-k "test_cooccurrence_matrix_ or test_ppmi_"`).

Dans tes notes : pourquoi la ligne d'un mot très fréquent (« the ») domine-t-elle la matrice de comptes, et que fait la PPMI de ses cases ? Que vaut la PPMI de deux mots qui ne se rencontrent jamais, et pourquoi la PMI, elle, ne conviendrait-elle pas à une matrice creuse ?

In [48]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")   # reload your saved file
with checked("B2.22"):
    Em = mylearn.embeddings
    ids_22 = np.array([0, 0, 0, 1], dtype=np.int32)              # int32, as the tests
    arg_22 = ids_22.copy()
    C_22 = Em.cooccurrence_matrix(arg_22, 2, window=1)
    if returned("B2.22", "cooccurrence_matrix", C_22):
        ok_22 = (isinstance(C_22, np.ndarray) and C_22.dtype.kind == "f" and C_22.shape == (2, 2)
                 and np.array_equal(C_22, [[4.0, 1.0], [1.0, 0.0]]))
        verdict("B2.22", ok_22, "[0, 0, 0, 1], fenêtre 1 : C[0, 0] = 4, chaque répétition d'une paire compte ; une matrice "
                "float (2, 2).",
                f"cooccurrence_matrix([0, 0, 0, 1], 2, window=1) doit rendre la matrice float [[4, 1], [1, 0]] (une paire "
                f"répétée s'ajoute à chaque fois) ; reçu {C_22!r}.")
        verdict("B2.22", np.array_equal(arg_22, ids_22), "les ids reçus n'ont pas changé.", "cooccurrence_matrix a modifié les ids reçus.")
    right_22 = Em.cooccurrence_matrix([2, 0, 1], 3, window=2, weighting="harmonic", symmetric=False)
    same("B2.22", "[2, 0, 1], fenêtre 2, harmonique, contextes de droite seulement", right_22,
         [[0, 1, 0], [0, 0, 0], [1, 0.5, 0]], rtol=1e-6, atol=1e-6)
    # Holmes: the 2 000 most frequent words (the others removed here), against a count by scipy.sparse
    import scipy.sparse as sp
    stoi_22, _ = wbtext.top_vocab(WORDS_H, 2000)
    h_22 = wbtext.word_ids(WORDS_H[:30000], stoi_22)
    h_22 = h_22[h_22 >= 0]
    pairs_22 = [(h_22[:-d], h_22[d:], np.full(len(h_22) - d, 1.0 / d)) for d in (1, 2, 3)]
    rows_22 = np.concatenate([a for a, b, w in pairs_22] + [b for a, b, w in pairs_22])
    cols_22 = np.concatenate([b for a, b, w in pairs_22] + [a for a, b, w in pairs_22])
    want_22 = sp.coo_matrix((np.concatenate([w for *_, w in pairs_22] * 2), (rows_22, cols_22)), shape=(2000, 2000)).toarray()
    same("B2.22", f"Holmes ({len(h_22):_} mots, 2 000 formes), fenêtre 3, harmonique : la matrice d'un comptage "
         "par scipy.sparse".replace("_", " "), Em.cooccurrence_matrix(h_22, 2000, window=3, weighting="harmonic"), want_22,
         rtol=1e-6, atol=1e-6)
    for label_22, call_22 in (("des ids 2-D", lambda: Em.cooccurrence_matrix([[0, 1]], 2)),
                              ("window = 0", lambda: Em.cooccurrence_matrix([0, 1], 2, window=0)),
                              ("weighting = 'gaussian'", lambda: Em.cooccurrence_matrix([0, 1], 2, weighting="gaussian")),
                              ("un id égal à vocab_size", lambda: Em.cooccurrence_matrix([0, 2], 2)),
                              ("un id de -1", lambda: Em.cooccurrence_matrix([0, -1], 2))):
        refuses("B2.22", label_22, call_22)

    # ppmi: the matrix chien / chat / os of the fiche (§202.8), a word never seen, an « independent » matrix
    C3_22 = np.array([[0, 4, 1], [4, 0, 1], [1, 1, 2]])          # integers: ppmi must accept them
    arg3_22 = C3_22.copy()
    P_22 = Em.ppmi(arg3_22)
    if returned("B2.22", "ppmi", P_22):
        verdict("B2.22", isinstance(P_22, np.ndarray) and P_22.shape == (3, 3) and np.isclose(P_22[0, 1], math.log(56 / 25))
                and P_22[0, 2] == 0 and np.isclose(P_22[2, 2], math.log(28 / 16)),
                "PPMI(chien, chat) = log 2,24 ≈ 0,807, PPMI(chien, os) = 0 (PMI négative), PPMI(os, os) ≈ 0,560, comme la fiche.",
                f"sur la matrice chien / chat / os de la fiche, attendu PPMI(chien, chat) ≈ 0,807, PPMI(chien, os) = 0 et "
                f"PPMI(os, os) ≈ 0,560 (logarithme naturel) ; reçu {np.round(np.asarray(P_22, dtype=float), 3).tolist()}.")
        verdict("B2.22", np.array_equal(arg3_22, C3_22), "la matrice reçue n'a pas changé.", "ppmi a modifié la matrice reçue.")
    P75_22 = Em.ppmi(C3_22, alpha=0.75)
    verdict("B2.22", np.isclose(P75_22[0, 1], 0.8230, atol=5e-4), f"alpha = 0,75 : PPMI(chien, chat) = {fr(P75_22[0, 1])}, "
            "comme la fiche.", f"alpha = 0,75 : attendu PPMI(chien, chat) ≈ 0,823 (fiche §202.8) ; reçu {fr(P75_22[0, 1])} : "
            "le lissage ne porte que sur les colonnes, puis se renormalise.")
    with warnings.catch_warnings(record=True) as caught_22, np.errstate(all="warn"):
        warnings.simplefilter("always")
        Z_22 = Em.ppmi(np.array([[0.0, 2, 1, 0], [2, 0, 3, 0], [1, 3, 0, 0], [0, 0, 0, 0]]))
    verdict("B2.22", np.all(np.isfinite(Z_22)) and np.all(Z_22[3] == 0) and np.all(Z_22[:, 3] == 0),
            "un mot jamais vu (ligne et colonne nulles) : des 0, sans NaN.",
            "une ligne et une colonne nulles doivent donner des 0, jamais NaN : mets 0 là où C == 0.")
    if any(issubclass(w.category, RuntimeWarning) for w in caught_22):
        print("💡 Ex B2.22 : ppmi émet un avertissement de NumPy (log(0) ou 0 / 0) : calcule le logarithme sur les seules "
              "cases non nulles, ou dans np.errstate(divide='ignore', invalid='ignore').")
    indep_22 = Em.ppmi(np.outer([1.0, 2, 3], [4.0, 1, 5]))
    verdict("B2.22", np.allclose(indep_22, 0, atol=1e-9), "une matrice « indépendante » np.outer(u, v) : PPMI nulle partout.",
            f"np.outer(u, v) donne p(w, c) = p(w) p(c) : attendu une PPMI nulle ; reçu le maximum {float(np.max(indep_22)):.3g}.")
    for label_22, call_22 in (("une matrice 1-D", lambda: Em.ppmi([1.0, 2.0])),
                              ("une entrée négative", lambda: Em.ppmi([[1.0, -1.0], [0.0, 2.0]])),
                              ("une somme nulle", lambda: Em.ppmi(np.zeros((2, 2))))):
        refuses("B2.22", label_22, call_22)

ℹ️ Ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2.
⏳ Ex B2.22 : pas encore fait (ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2).


In [49]:
if written(lambda: mylearn.embeddings.cooccurrence_matrix([0, 1], 2, window=1)):
    run_pytest_group(TEST_EMB, "test_cooccurrence_matrix_ or test_ppmi_")   # 36 tests
else:
    print("⏳ Ex B2.22 : les tests attendent tes fonctions.")

⏳ Ex B2.22 : les tests attendent tes fonctions.


### Ex B2.23 — cosine_similarity_matrix et most_similar 🔨 ★★ ⏱️ 20 min
**Objectif :** calculer tous les cosinus entre deux ensembles de vecteurs, puis chercher les plus proches voisins d'une requête, égalités et exclusions comprises.  
**Prérequis :** ✏️ B2.2 · 0B (produit scalaire, norme) · fiche §202.9 (« Le cosinus », « Les plus proches voisins », « Les analogies ») · **Fil rouge :** Holmes · **mylearn :** `embeddings.py` · **Parcours :** R, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/embeddings.py` (créé par `python tools/start_chapter.py B2`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb2_embeddings.py -q` les lance tous dans un terminal.

**Contrat**
- `cosine_similarity_matrix(A, B=None, eps=1e-12)` :
  - rend `S` de forme `(n, m)`, `S[i, j] = A[i] · B[j] / ((‖A[i]‖ + eps)(‖B[j]‖ + eps))` : `eps` s'ajoute à **chaque** norme, pas sous la racine ni au produit (testé avec `eps = 0,5` et 2) ; une ligne nulle a un cosinus de 0 avec tout ;
  - `B = None` vaut `B = A` ; accepte des listes et des entiers, sans les modifier ;
  - `ValueError` pour `A` ou `B` qui n'est pas 2-D, et pour des nombres de colonnes différents.
- `most_similar(E, query, k=10, exclude=())` :
  - rend un **tuple de deux tableaux NumPy** 1-D : les indices (entiers) et les cosinus (flottants), par cosinus décroissant ;
  - les égalités vont au **plus petit indice** (testé sur 120 lignes, dont 40 copies de trois lignes : le tri par défaut de `np.argsort` et `np.argpartition` ne le garantissent pas) ;
  - les lignes de `exclude` (des indices valides de `E`) sont retirées **avant** de garder les `k` premières ; s'il en reste moins de `k`, toutes ; si tout est exclu, deux tableaux de forme `(0,)` ;
  - une requête nulle donne des cosinus nuls, donc les `k` premiers indices ;
  - `ValueError` pour `k < 1`, une requête qui n'a pas la longueur `E.shape[1]`, ou de forme `(1, d)`.

La première cellule de vérification compare ta matrice à `sklearn.metrics.pairwise.cosine_similarity`, essaie `eps`, puis `most_similar` sur 60 lignes faites de 20 copies de trois directions (égalités), avec des exclusions, trop peu de lignes, une requête nulle et trois refus. La seconde lance les 26 tests de l'exercice (`-k "test_cosine_similarity_matrix_ or test_most_similar_"`).

Dans tes notes : pourquoi normaliser les lignes **une fois**, puis faire un seul produit matriciel, plutôt que de calculer chaque cosinus séparément ? Pour l'analogie du ✏️ B2.2, quels indices passes-tu à `exclude`, et que se passerait-il sans eux ?

In [50]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")   # reload your saved file
with checked("B2.23"):
    Em = mylearn.embeddings
    rng_23 = np.random.default_rng(23)
    A_23, B_23 = rng_23.normal(size=(5, 3)), rng_23.normal(size=(4, 3))
    A_23[2] = 0.0                                                   # a row of zeros
    argA_23, argB_23 = A_23.copy(), B_23.copy()
    S_23 = Em.cosine_similarity_matrix(argA_23, argB_23)
    from sklearn.metrics.pairwise import cosine_similarity
    if returned("B2.23", "cosine_similarity_matrix", S_23):
        same("B2.23", "cosine_similarity_matrix(A, B), A (5, 3) avec une ligne nulle, B (4, 3), contre scikit-learn",
             S_23, cosine_similarity(A_23, B_23), rtol=1e-6, atol=1e-9)
        verdict("B2.23", np.array_equal(argA_23, A_23) and np.array_equal(argB_23, B_23), "A et B n'ont pas changé.",
                "cosine_similarity_matrix a modifié A ou B.")
    same("B2.23", "cosine_similarity_matrix(A) (B = None : B = A)", Em.cosine_similarity_matrix(A_23), cosine_similarity(A_23),
         rtol=1e-6, atol=1e-9)
    same("B2.23", "[[3, 4]] contre [[6, 8]] avec eps = 0,5 (ajouté à chaque norme)",
         Em.cosine_similarity_matrix([[3, 4]], [[6, 8]], eps=0.5), [[50 / (5.5 * 10.5)]])
    refuses("B2.23", "A 1-D", lambda: Em.cosine_similarity_matrix([1.0, 2.0]))
    refuses("B2.23", "A et B de 3 et 2 colonnes", lambda: Em.cosine_similarity_matrix(A_23, B_23[:, :2]))

    E_23 = np.tile(np.eye(3), (20, 1))                              # 60 rows: 20 copies of three directions
    out_23 = Em.most_similar(E_23, [1.0, 0.0, 0.0], k=5)
    if returned("B2.23", "most_similar", out_23):
        idx_23, sims_23 = pair("B2.23", "most_similar", out_23, names=("indices", "similarities"))
        verdict("B2.23", isinstance(idx_23, np.ndarray) and isinstance(sims_23, np.ndarray)
                and np.issubdtype(idx_23.dtype, np.integer) and idx_23.tolist() == [0, 3, 6, 9, 12]
                and np.allclose(sims_23, 1),
                "20 lignes à égalité : les 5 plus petits indices, [0, 3, 6, 9, 12] (un tri stable), en tableaux NumPy.",
                f"20 lignes égales à la requête : attendu les indices [0, 3, 6, 9, 12] (égalités : le plus petit indice), "
                f"deux tableaux NumPy ; reçu {idx_23!r}.")
    idx_23, _ = Em.most_similar(E_23, [1.0, 0.0, 0.0], k=3, exclude=(0, 3))
    verdict("B2.23", list(idx_23) == [6, 9, 12], "exclude retiré avant le top-k : [6, 9, 12].",
            f"avec exclude=(0, 3), attendu [6, 9, 12] (on retire avant de garder les k premiers) ; reçu {list(idx_23)}.")
    few_23 = Em.most_similar(np.eye(3), [1.0, 1.0, 0.0], k=10, exclude=(1,))
    verdict("B2.23", list(few_23[0]) == [0, 2], "moins de k lignes restantes : toutes, ici [0, 2].",
            f"3 lignes, une exclue, k = 10 : attendu les deux restantes [0, 2] ; reçu {list(few_23[0])}.")
    none_23 = Em.most_similar(np.eye(2), [1.0, 0.0], k=2, exclude=(0, 1))
    verdict("B2.23", none_23[0].shape == (0,) and none_23[1].shape == (0,), "tout exclu : deux tableaux de forme (0,).",
            f"tout exclu : attendu deux tableaux vides de forme (0,) ; reçu {none_23!r}.")
    zero_23 = Em.most_similar(np.eye(3), [0.0, 0.0, 0.0], k=2)
    verdict("B2.23", list(zero_23[0]) == [0, 1] and np.allclose(zero_23[1], 0),
            "une requête nulle : des similarités nulles et les k premiers indices.",
            f"une requête nulle doit donner des similarités nulles et les indices [0, 1] ; reçu {zero_23!r}.")
    refuses("B2.23", "k = 0", lambda: Em.most_similar(np.eye(3), [1.0, 0, 0], k=0))
    refuses("B2.23", "une requête de longueur 2 pour des lignes de 3", lambda: Em.most_similar(np.eye(3), [1.0, 0], k=1))
    refuses("B2.23", "une requête de forme (1, 3)", lambda: Em.most_similar(np.eye(3), [[1.0, 0, 0]], k=1))

ℹ️ Ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2.
⏳ Ex B2.23 : pas encore fait (ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2).


In [51]:
if written(lambda: mylearn.embeddings.cosine_similarity_matrix([[1.0, 0.0]])):
    run_pytest_group(TEST_EMB, "test_cosine_similarity_matrix_ or test_most_similar_")   # 26 tests
else:
    print("⏳ Ex B2.23 : les tests attendent tes fonctions.")

⏳ Ex B2.23 : les tests attendent tes fonctions.


### Ex B2.24 — Embeddings PPMI + SVD sur Holmes : les voisins de « Holmes », « Watson » et « letter » 📦 ★★ ⏱️ 30 min
**Objectif :** construire des vecteurs de mots par comptage, PPMI et SVD tronquée, avec ta librairie et scikit-learn, puis les interroger.  
**Prérequis :** Ex B2.22, B2.23 · ch. 12 (PCA) · fiche §202.8 (« Réduire : la SVD tronquée »), §202.9 · **Fil rouge :** Holmes · **Parcours :** C

La carte de la fiche (§202.9) a été faite avec des vecteurs de Holmes construits par comptage. Tu refais le calcul avec **ta** librairie, et tu interroges trois mots dont la fiche ne dit rien.

**Contrat**
- `ppmi_svd_24(words, n_words=4000, window=2, dim=100, alpha=0.75)` rend `(E, stoi, itos)` :
  - le vocabulaire est `wbtext.top_vocab(words, n_words)`, les `n_words` mots les plus fréquents (`itos[0]` est le plus fréquent) ;
  - les co-occurrences viennent de **ton** `cooccurrence_matrix` (fenêtre symétrique, pondération uniforme), la PPMI lissée de **ton** `ppmi(C, alpha)` ;
  - un mot **hors du vocabulaire garde sa place** dans le texte, mais ses paires ne comptent pas : on ne le retire pas, sinon deux mots qui l'entourent deviendraient voisins. `wbtext.word_ids(words, stoi)` lui donne l'id −1, que ton `cooccurrence_matrix` refuse : à toi de trouver comment ;
  - la SVD tronquée est `TruncatedSVD(n_components=dim, algorithm="arpack", random_state=0)` de scikit-learn, sur la matrice PPMI ;
  - `E` est `(n_words, dim)` : les lignes de $\mathbf{U}_d\,\mathbf{S}_d^{1/2}$ (pas de `fit_transform` seul, qui rend $\mathbf{U}_d\,\mathbf{S}_d$), chacune ramenée à la **norme 1**.
- `neighbours_24(E, stoi, itos, word, k=8)` : la liste des `k` couples `(mot, cosinus)` les plus proches de `word`, par cosinus décroissant, sans `word` lui-même, trouvés par **ton** `most_similar`.

La cellule de vérification lance ta fonction sur les 105 863 mots de Holmes (5 à 15 s, la SVD comprise), puis compare les cosinus de « holmes », « watson » et « letter » avec les 4 000 mots à ceux d'une référence (le même calcul, compté par `scipy.sparse`, `wbtext.reference_ppmi_svd`). Elle affiche enfin les 8 voisins des trois mots.

Dans tes notes :
- les voisins de « holmes » : de quelle nature sont-ils (des noms, des mots qui l'accompagnent dans le texte, des verbes) ? Pourquoi « sherlock » est-il si proche ?
- ceux de « watson » et de « letter » : lesquels ont un sens voisin, lesquels partagent seulement des contextes (« a … of », « the … was ») ?
- « watson » n'apparaît que 81 fois, « letter » 30 fois : que dit la qualité de leurs voisins de la quantité de texte qu'il faut ?

In [52]:
from sklearn.decomposition import TruncatedSVD


def ppmi_svd_24(words, n_words=4000, window=2, dim=100, alpha=0.75):
    """(E, stoi, itos): PPMI + SVD word vectors (rows of norm 1) of the n_words most frequent words of `words`, made
    with your cooccurrence_matrix and ppmi; a word outside the vocabulary keeps its place, but its pairs do not count."""
    raise NotImplementedError("ppmi_svd_24")


def neighbours_24(E, stoi, itos, word, k=8):
    """The k nearest other words of `word` by cosine, with your most_similar: [(word, cosine), ...]."""
    raise NotImplementedError("neighbours_24")

In [53]:
WORDS_24 = ("holmes", "watson", "letter")
ok_24 = False
with checked("B2.24"):
    start_24 = time.perf_counter()
    out_24 = ppmi_svd_24(WORDS_H)
    seconds_24 = time.perf_counter() - start_24
    if not returned("B2.24", "ppmi_svd_24", out_24):
        raise StopCheck
    E_24, stoi_24, itos_24 = pair("B2.24", "ppmi_svd_24", out_24, names=("E", "stoi", "itos"))
    E_24 = np.asarray(E_24, dtype=float)
    verdict("B2.24", E_24.shape == (4000, 100) and np.allclose(np.linalg.norm(E_24, axis=1), 1, atol=1e-6)
            and list(itos_24) == wbtext.top_vocab(WORDS_H, 4000)[1],
            f"E : 4 000 mots × 100 dimensions, lignes de norme 1, le vocabulaire de top_vocab ({fr(seconds_24, 1)} s).",
            f"attendu E de forme (4000, 100) aux lignes de norme 1 et itos = top_vocab(words, 4000)[1] ; reçu E {E_24.shape}.")
    R_24, rstoi_24, _ = wbtext.reference_ppmi_svd(WORDS_H)     # the same pipeline, counted by scipy.sparse
    rows_24 = [stoi_24[w] for w in WORDS_24]
    gap_24 = float(np.abs(E_24[rows_24] @ E_24.T - R_24[rows_24] @ R_24.T).max())
    verdict("B2.24", gap_24 < 1e-3, "les cosinus de « holmes », « watson » et « letter » avec les 4 000 mots sont ceux de "
            f"la référence (écart maximal {gap_24:.1e}).",
            f"les cosinus de « holmes », « watson » et « letter » s'écartent de la référence jusqu'à {gap_24:.2f} : as-tu "
            "gardé la place des mots hors vocabulaire (sans les retirer du texte), lissé les seules colonnes (alpha), pris "
            "U S^(1/2) (pas U S), normalisé les lignes, et algorithm='arpack' ?")
    nb_24 = {w: neighbours_24(E_24, stoi_24, itos_24, w) for w in WORDS_24}
    if not returned("B2.24", "neighbours_24", nb_24["holmes"]):
        raise StopCheck
    want_24 = {w: nearest_words(E_24, list(itos_24), w, 8, dict(stoi_24)) for w in WORDS_24}
    verdict("B2.24", all(len(nb_24[w]) == 8 and [x for x, _ in nb_24[w]] == [x for x, _ in want_24[w]] for w in WORDS_24),
            "neighbours_24 : les 8 voisins de chaque mot, sans le mot lui-même, par cosinus décroissant.",
            "neighbours_24 doit rendre les 8 plus proches autres mots, (mot, cosinus) par cosinus décroissant, sans le mot.")
    for w in WORDS_24:
        print(f"{w:>7} : " + ", ".join(f"{x} {fr(s, 2)}" for x, s in nb_24[w]))
    ok_24 = True

⏳ Ex B2.24 : pas encore fait (ppmi_svd_24).


### Ex B2.25 — word2vec (skip-gram, échantillonnage négatif) en PyTorch sur Verne 📦 ★★★ ⏱️ 60 min
**Objectif :** entraîner des embeddings word2vec en PyTorch, avec les trois réglages de l'article : la loi des négatifs, leur nombre et le sous-échantillonnage.  
**Prérequis :** ∂ B2.7 · Ex B2.24 · ch. 20 (boucle d'entraînement PyTorch) · fiche §202.7, §202.8 (« word2vec : prédire plutôt que compter ») · **Fil rouge :** Verne · **Parcours :** C

Tu entraînes un skip-gram avec échantillonnage négatif (fiche §202.8) sur *Le Tour du monde en quatre-vingts jours*. La cellule suivante prépare les données : les 1 590 mots vus au moins 5 fois (`STOI_25`, `ITOS_25`, ids par fréquence décroissante), le texte `IDS_25` **sans** ses mots rares (word2vec les retire avant l'entraînement : contrairement à B2.24, ses voisins se rapprochent), et leurs comptes `COUNTS_25`.

**Contrat**
- `keep_probability_25(counts, t=1e-3)` : pour chaque mot, la probabilité de **garder** une de ses occurrences, $\min\big(1, \sqrt{t / f(w)}\big)$ avec $f(w) = \#(w) / \sum \#$ (la fiche donne la probabilité de la jeter, $1 - \sqrt{t/f}$, avec $t \approx 10^{-5}$ pour un milliard de mots ; sur un roman de 72 000 mots, $t = 10^{-3}$) ;
- `noise_distribution_25(counts, power=0.75)` : la loi des négatifs, les comptes à la puissance `power`, de somme 1 ;
- `skipgram_pairs_25(ids, window=2)` : `(centers, contexts)`, deux tableaux `int64` de même longueur, une entrée par couple de positions $(i, j)$ avec $1 \le |i - j| \le$ `window` (le mot `ids[i]` et son contexte `ids[j]`), dans l'ordre que tu veux ;
- `SkipGram25(vocab_size, dim)`, un `nn.Module` :
  - deux tables `nn.Embedding(vocab_size, dim)` : `self.center` (les $\mathbf{v}_w$), initialisée uniforme dans $[-0{,}5/d ;\ 0{,}5/d]$, et `self.context` (les $\mathbf{u}_c$), initialisée à **zéro**, comme word2vec ;
  - `forward(centers, contexts, negatives)`, avec des ids `(B,)`, `(B,)` et `(B, K)`, rend la **moyenne sur le batch** de la loss de la fiche, $-\log \sigma(\mathbf{u}_c^\top \mathbf{v}_w) - \sum_k \log \sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_w)$, un tenseur scalaire, calculée avec `F.logsigmoid` ;
- `train_word2vec_25(ids, counts, dim=100, window=2, negatives=5, t=1e-3, epochs=EPOCHS_25, batch_size=512, lr=0.01, seed=0)` rend `(model, losses)` : un `SkipGram25` entraîné par `torch.optim.Adam(lr)`, et la loss moyenne de chaque epoch (une liste de `float`). À chaque epoch : un **nouveau** sous-échantillonnage (`np.random.default_rng(seed)`, une fois au départ, avec `torch.manual_seed(seed)`), les paires du texte gardé, mélangées, des mini-batches de `batch_size` paires, et pour chaque paire `negatives` négatifs tirés selon `noise_distribution_25` (`torch.multinomial`, avec remise).

La première cellule de vérification essaie tes fonctions sur des cas visibles, compare `forward` à la formule sur des poids fixés, et lance un mini-entraînement. La suivante entraîne ton word2vec sur tout le texte (`EPOCHS_25` = 10 epochs en mode rapide, ≈ 10 s sur deux cœurs ; 30 en mode complet), garde les vecteurs `E_25` (les lignes de `center`, de norme 1) et mesure leur qualité : pour douze paires de mots liés (« deux » et « trois », « bombay » et « calcutta »…), le rang du second parmi les voisins du premier.

Dans tes notes : au premier pas, `context` est nul : que valent alors tous les scores, et la loss d'une paire avec 5 négatifs (calcule-la, puis compare à ta loss de la première epoch) ? Que changerait un sous-échantillonnage avec $t = 10^{-5}$ sur ce petit roman (calcule la probabilité de garder « fogg ») ? Pourquoi garde-t-on la table `center`, et pas `context` ?

In [54]:
# The data of word2vec: the words of Verne seen at least 5 times (the rarer ones are removed from the text, as word2vec
# does), their ids by decreasing frequency, and their counts
MIN_COUNT_25 = 5
N_VOCAB_25 = sum(1 for n in Counter(WORDS_V).values() if n >= MIN_COUNT_25)
STOI_25, ITOS_25 = wbtext.top_vocab(WORDS_V, N_VOCAB_25)       # id 0 = « de », the most frequent
IDS_25 = wbtext.word_ids(WORDS_V, STOI_25)
IDS_25 = IDS_25[IDS_25 >= 0]                                   # the text without its rare words (int64)
COUNTS_25 = np.bincount(IDS_25, minlength=N_VOCAB_25).astype(float)
EPOCHS_25 = 10 if FAST_MODE else 30
# pairs of related words, to measure the quality of the vectors: the rank of the second among the neighbours of the first
PAIRS_25 = [("deux", "trois"), ("quatre", "cinq"), ("vingt", "cent"), ("heures", "minutes"), ("jours", "heures"),
            ("bombay", "calcutta"), ("yokohama", "hong"), ("train", "wagon"), ("paquebot", "steamer"), ("main", "mains"),
            ("tête", "yeux"), ("dit", "répondit")]
print(f"{N_VOCAB_25:_} mots de vocabulaire, {len(IDS_25):_} mots dans le texte d'entraînement, {EPOCHS_25} epochs.".replace("_", " "))

1 590 mots de vocabulaire, 60 979 mots dans le texte d'entraînement, 10 epochs.


In [55]:
def keep_probability_25(counts, t=1e-3):
    """For each word of the vocabulary, the probability of KEEPING one occurrence: min(1, sqrt(t / f)), f = count / total
    (a float array of the shape of counts)."""
    raise NotImplementedError("keep_probability_25")


def noise_distribution_25(counts, power=0.75):
    """The law of the negatives: counts ** power, normalised to a sum of 1 (a float array)."""
    raise NotImplementedError("noise_distribution_25")


def skipgram_pairs_25(ids, window=2):
    """(centers, contexts): two int64 arrays of the same length, one entry per (position i, position j) of the sequence
    with 1 <= |i - j| <= window (the word ids[i] and its context ids[j])."""
    raise NotImplementedError("skipgram_pairs_25")


class SkipGram25(nn.Module):
    """Skip-gram with negative sampling: the table `center` (the vectors v_w) and the table `context` (the u_c)."""

    def __init__(self, vocab_size, dim):
        super().__init__()
        raise NotImplementedError("SkipGram25.__init__")

    def forward(self, centers, contexts, negatives):
        """The mean over the batch of the loss of the fiche: centers and contexts (B,), negatives (B, K) -> a scalar."""
        raise NotImplementedError("SkipGram25.forward")


def train_word2vec_25(ids, counts, dim=100, window=2, negatives=5, t=1e-3, epochs=EPOCHS_25, batch_size=512, lr=0.01,
                      seed=0):
    """(model, losses): a SkipGram25 trained by Adam on the text `ids`, and the mean loss of each epoch."""
    raise NotImplementedError("train_word2vec_25")

In [56]:
ok_25 = False
with checked("B2.25"):
    keep_25 = keep_probability_25(np.array([990.0, 10.0]), t=1e-3)
    if returned("B2.25", "keep_probability_25", keep_25):
        same("B2.25", "keep_probability_25([990, 10], t=1e-3) (f = 0,99 et 0,01)", keep_25,
             [math.sqrt(1e-3 / 0.99), math.sqrt(1e-3 / 0.01)], rtol=1e-6)
    verdict("B2.25", np.allclose(keep_probability_25(np.array([1.0, 1e6]), t=1e-3), [1.0, math.sqrt(1e-3 / (1e6 / (1e6 + 1)))]),
            "un mot très rare est toujours gardé (probabilité 1, pas plus).",
            "la probabilité de garder un mot rare doit être bornée à 1 : min(1, sqrt(t / f)).")
    same("B2.25", "noise_distribution_25([100, 1]) (fiche §202.8)", noise_distribution_25(np.array([100.0, 1.0])),
         np.array([100 ** 0.75, 1.0]) / (100 ** 0.75 + 1), rtol=1e-6)
    pairs_25 = skipgram_pairs_25(np.array([5, 6, 7, 8]), window=2)
    centers_25, contexts_25 = pair("B2.25", "skipgram_pairs_25", pairs_25, names=("centers", "contexts"))
    want_pairs_25 = sorted((a, b) for i, a in enumerate([5, 6, 7, 8]) for j, b in enumerate([5, 6, 7, 8]) if 1 <= abs(i - j) <= 2)
    got_pairs_25 = sorted(zip(np.asarray(centers_25).tolist(), np.asarray(contexts_25).tolist()))
    verdict("B2.25", got_pairs_25 == want_pairs_25 and np.asarray(centers_25).dtype == np.int64,
            f"skipgram_pairs_25([5, 6, 7, 8], window=2) : les {len(want_pairs_25)} paires (mot, contexte), en int64.",
            f"skipgram_pairs_25([5, 6, 7, 8], window=2) : attendu les {len(want_pairs_25)} paires {want_pairs_25} (dans "
            f"n'importe quel ordre, en int64) ; reçu {got_pairs_25}.")
    toy_25 = SkipGram25(10, 4)
    verdict("B2.25", isinstance(getattr(toy_25, "center", None), nn.Embedding) and isinstance(getattr(toy_25, "context", None), nn.Embedding)
            and toy_25.center.weight.shape == (10, 4) and toy_25.context.weight.shape == (10, 4)
            and toy_25.context.weight.detach().abs().max().item() == 0 and toy_25.center.weight.detach().abs().max().item() <= 0.5 / 4,
            "SkipGram25(10, 4) : deux nn.Embedding (10, 4), center dans [-0,125 ; 0,125] et context à 0 au départ.",
            "SkipGram25 doit avoir deux nn.Embedding(vocab_size, dim), center (uniforme dans [-0,5/d ; 0,5/d]) et "
            "context (des zéros), comme word2vec.")
    with torch.no_grad():                                        # fixed weights: the loss of the fiche, by hand
        g_25 = torch.Generator().manual_seed(25)
        toy_25.center.weight.copy_(torch.randn(10, 4, generator=g_25))
        toy_25.context.weight.copy_(torch.randn(10, 4, generator=g_25))
    c_25, x_25, n_25 = torch.tensor([1, 4, 4]), torch.tensor([2, 0, 7]), torch.tensor([[3, 5], [9, 9], [1, 2]])
    loss_25 = toy_25(c_25, x_25, n_25)
    V_25, U_25 = toy_25.center.weight.detach(), toy_25.context.weight.detach()
    want_25 = float(np.mean([-F.logsigmoid(U_25[x] @ V_25[c]).item() - sum(F.logsigmoid(-(U_25[n] @ V_25[c])).item() for n in ns)
                             for c, x, ns in zip(c_25.tolist(), x_25.tolist(), n_25.tolist())]))
    verdict("B2.25", isinstance(loss_25, torch.Tensor) and loss_25.ndim == 0 and math.isclose(loss_25.item(), want_25, rel_tol=1e-5),
            f"forward : la moyenne sur le batch de la loss de la fiche ({fr(want_25)} sur trois paires et K = 2 négatifs).",
            f"forward doit rendre un tenseur scalaire, la moyenne sur le batch de -log σ(u_c·v_w) - Σ log σ(-u_n·v_w) : "
            f"attendu {fr(want_25)} ; reçu {loss_25!r}.")
    small_25 = train_word2vec_25(IDS_25[:3000], COUNTS_25, dim=8, epochs=2, seed=1)
    model_s25, losses_s25 = pair("B2.25", "train_word2vec_25", small_25, names=("model", "losses"))
    verdict("B2.25", isinstance(model_s25, SkipGram25) and len(losses_s25) == 2 and all(isinstance(v, float) for v in losses_s25),
            "train_word2vec_25 rend (le modèle SkipGram25, la loss moyenne de chaque epoch, des float).",
            "train_word2vec_25 doit rendre le couple (un SkipGram25, une liste d'une loss float par epoch).")
    ok_25 = True

⏳ Ex B2.25 : pas encore fait (keep_probability_25).


In [57]:
if not globals().get("ok_25"):
    print("⏳ Ex B2.25 : l'entraînement attend que tes fonctions passent la cellule précédente.")
else:
    with checked("B2.25"):
        ok_25 = False
        start_25 = time.perf_counter()
        MODEL_25, LOSSES_25 = train_word2vec_25(IDS_25, COUNTS_25, seed=0)
        seconds_25 = time.perf_counter() - start_25
        E_25 = MODEL_25.center.weight.detach().double().numpy()
        E_25 = E_25 / np.linalg.norm(E_25, axis=1, keepdims=True)      # the word vectors: rows of norm 1
        print(f"{EPOCHS_25} epochs en {fr(seconds_25, 1)} s ; loss moyenne par epoch : "
              + ", ".join(fr(v, 3) for v in LOSSES_25))
        verdict("B2.25", len(LOSSES_25) == EPOCHS_25 and LOSSES_25[-1] < LOSSES_25[0] - 0.3,
                f"la loss baisse de {fr(LOSSES_25[0], 2)} à {fr(LOSSES_25[-1], 2)}.",
                f"la loss devrait baisser nettement d'une epoch à l'autre ({fr(LOSSES_25[0], 2)} → {fr(LOSSES_25[-1], 2)}) : "
                "relis les signes de forward, l'ordre zero_grad / backward / step, et le learning rate.")
        ranks_25 = []
        for a, b in PAIRS_25:
            order = [w for w, _ in nearest_words(E_25, ITOS_25, a, len(ITOS_25) - 1, STOI_25)]
            ranks_25.append(order.index(b) + 1)
        print("Rang du second mot parmi les voisins du premier : "
              + ", ".join(f"{a} → {b} {r}" for (a, b), r in zip(PAIRS_25, ranks_25)))
        median_25 = float(np.median(ranks_25))
        verdict("B2.25", median_25 <= 100, f"rang médian {median_25:g} sur {len(ITOS_25) - 1} voisins possibles (au hasard : "
                f"≈ {len(ITOS_25) // 2}) : tes vecteurs rapprochent des mots de même sens.",
                f"rang médian {median_25:g} (au hasard : ≈ {len(ITOS_25) // 2}) : tes vecteurs n'ont pas appris grand-chose. "
                "Relis le sous-échantillonnage, la loi des négatifs et forward.")
        ok_25 = median_25 <= 100

⏳ Ex B2.25 : l'entraînement attend que tes fonctions passent la cellule précédente.


### Ex B2.26 — Prédire les voisins de « Fogg » et de « Passepartout » avant de les calculer 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que capturent les voisins de deux personnages dans des vecteurs word2vec, avant de les calculer.  
**Prérequis :** Ex B2.25 · fiche §202.8 (l'hypothèse distributionnelle), §202.9 (« Les plus proches voisins ») · **Fil rouge :** Verne · **Parcours :** C

Phileas Fogg (« fogg » : 685 occurrences, le 14ᵉ mot du roman) et Passepartout (451) sont les deux héros du roman. Tes vecteurs `E_25` de B2.25 vont donner leurs plus proches voisins par cosinus. **Avant d'exécuter**, écris tes deux prédictions :
- a) `fogg_26` : les **deux** plus proches voisins de « fogg » sont… `"A"` deux autres personnages (passepartout, fix, aouda…) ; `"B"` « phileas » et « mr », dans un ordre ou l'autre ; `"C"` deux verbes de parole (« dit », « répondit »…) ; `"D"` deux autres mots ;
- b) `characters_26` : `True` ou `False` : au moins 5 des 10 plus proches voisins de « passepartout » sont des noms de personnages du roman (la liste `CHARACTERS_26` de la cellule suivante).

L'expérience n'affiche rien tant que tes deux prédictions ne sont pas écrites, et attend ton word2vec de B2.25 ; elle vérifie tes prédictions à la fin.

In [58]:
# The names of the characters of the novel (titles such as « mr » are not names)
CHARACTERS_26 = {"fogg", "phileas", "passepartout", "fix", "aouda", "proctor", "stamp", "cromarty", "francis", "obadiah",
                 "speedy", "batulcar", "stuart", "flanagan", "sullivan", "fallentin", "ralph", "gauthier", "andrew", "john",
                 "mudge", "bunsby"}

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [59]:
fogg_26 = ...         # a) the TWO nearest neighbours of « fogg »: "A" two other characters (passepartout, fix, aouda…),
                      #    "B" « phileas » and « mr », "C" two verbs of speech (dit, répondit…), "D" two other words
characters_26 = ...   # b) True or False: at least 5 of the 10 nearest neighbours of « passepartout » are names of
                      #    characters of the novel (CHARACTERS_26)

In [60]:
if any(answer is ... or answer is None for answer in [fogg_26, characters_26]):
    print("⏳ Ex B2.26 : écris d'abord tes deux prédictions dans la cellule précédente.")
else:
    if not globals().get("ok_25"):
        print("⏳ Ex B2.26 : l'expérience attend ton word2vec entraîné en B2.25 (les vecteurs E_25) ; tes prédictions seront "
              "vérifiées ensuite.")
    else:
        neighbours_26 = {w: nearest_words(E_25, ITOS_25, w, 10, STOI_25) for w in ("fogg", "passepartout")}
        for w, nb in neighbours_26.items():
            print(f"{w} ({int(COUNTS_25[STOI_25[w]])} fois) : " + ", ".join(f"{x} {fr(s, 2)}" for x, s in nb))
        top2_26 = {x for x, _ in neighbours_26["fogg"][:2]}
        names_26 = sum(x in CHARACTERS_26 for x, _ in neighbours_26["passepartout"])
        print(f"Les deux premiers voisins de « fogg » : {sorted(top2_26)} ; noms de personnages parmi les 10 voisins de "
              f"« passepartout » : {names_26}.")
        print("Fréquence (rang) des voisins de « passepartout » : "
              + ", ".join(f"{x} {STOI_25[x]}" for x, _ in neighbours_26["passepartout"]))
        wb.check("B2.26a", fogg_26)
        wb.check("B2.26b", characters_26)

⏳ Ex B2.26 : écris d'abord tes deux prédictions dans la cellule précédente.


Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Deux mots sont proches quand ils ont des **contextes** semblables (§202.8) : quels contextes « fogg » partage-t-il avec ses deux premiers voisins ? Regarde la fréquence (le rang) des voisins de « passepartout » : que remarques-tu, et qu'est-ce que cela dit de ce que mesure le cosinus de word2vec sur un petit texte ?

### Ex B2.27 — Lire une carte PCA/UMAP d'embeddings : groupes, axes et effet de la fréquence 📈 ★★ ⏱️ 20 min
**Objectif :** lire trois cartes en deux dimensions de vecteurs word2vec, et distinguer ce qu'elles montrent du sens et de la fréquence des mots.  
**Prérequis :** Ex B2.25 · ch. 12 (PCA, UMAP) · fiche §202.9 (« Visualiser ») · **Fil rouge :** Verne · **Parcours :** C

La cellule suivante dessine trois cartes de tes vecteurs `E_25` de B2.25 (≈ 20 s la première fois, le temps qu'UMAP se compile) :
1. la **PCA** des 1 590 mots, chaque point coloré par le logarithme de sa fréquence (violet : rare, jaune : fréquent) ;
2. l'**UMAP** des mêmes mots (`n_neighbors=15`, distance cosinus, graine fixée), avec les mêmes couleurs ;
3. la **PCA** des seuls mots de cinq groupes (nombres, temps, lieux, transports, parties du corps), chacun avec son nom, comme la carte de la fiche.

Lis les cartes et écris :
- a) `pca_axis_27` : `True` ou `False` : sur la carte (1), la couleur change régulièrement le long de l'axe **horizontal**, les mots fréquents d'un côté et les rares de l'autre ;
- b) `umap_frequent_27` : `True` ou `False` : sur la carte (2), les mots les plus fréquents (en jaune) se rassemblent dans une région (ou quelques-unes), au lieu d'être répartis sur toute la carte.

Une fois tes deux réponses écrites, la cellule de vérification affiche des mesures faites sur les coordonnées : la corrélation de rang entre la fréquence et chaque axe de la carte (1), la part des voisins des mots fréquents qui sont eux aussi fréquents sur la carte (2), et, pour chaque groupe de la carte (3), la part de ses voisins qui sont du même groupe.

Dans tes notes : sur la carte (3), quels groupes se séparent, lesquels se mêlent, et quels mots sont « mal placés » (pense aux contextes qu'ils partagent, comme « minutes » et les nombres dans la fiche) ? Pourquoi la fréquence se voit-elle sur la carte (2) et presque pas le long des axes de la (1) ? Pourquoi ne faut-il pas lire les **distances entre** les groupes d'un UMAP (ch. 12, B1) ?

In [61]:
# Five groups of words of Verne, for the third map
GROUPS_27 = {"nombres": "deux trois quatre cinq six sept huit neuf dix vingt cent mille".split(),
             "temps": "heures jours minutes heure jour".split(),
             "lieux": "bombay calcutta londres yokohama hong kong san francisco suez liverpool".split(),
             "transports": "mer paquebot train wagon locomotive traîneau steamer".split(),
             "corps": "mains main tête yeux bras pied pieds".split()}


def maps_27(E, counts, groups, seed=0):
    """The three maps of the figure: PCA of all the words, UMAP of all the words (None without umap-learn), PCA of the
    words of the groups."""
    from sklearn.decomposition import PCA
    pca_all = PCA(2, svd_solver="full").fit_transform(E)
    try:
        import umap
        umap_all = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=seed, n_jobs=1).fit_transform(E)
    except ImportError:
        umap_all = None
        print("⏳ Ex B2.27 : umap-learn n'est pas installé ici : la carte UMAP est « à valider sur Colab ».")
    words = [w for ws in groups.values() for w in ws]
    pca_groups = PCA(2, svd_solver="full").fit_transform(E[[STOI_25[w] for w in words]])
    return pca_all, umap_all, pca_groups, words


if not globals().get("ok_25"):
    print("⏳ Ex B2.27 : les cartes attendent ton word2vec entraîné en B2.25 (les vecteurs E_25).")
else:
    start_27 = time.perf_counter()
    PCA_ALL_27, UMAP_ALL_27, PCA_GROUPS_27, GROUP_WORDS_27 = maps_27(E_25, COUNTS_25, GROUPS_27)
    logf_27 = np.log10(COUNTS_25)
    fig_27, axes_27 = plt.subplots(1, 3, figsize=(16, 5))
    for ax, P, title in ((axes_27[0], PCA_ALL_27, "(1) PCA des 1 590 mots"), (axes_27[1], UMAP_ALL_27, "(2) UMAP des 1 590 mots")):
        if P is None:
            ax.set_axis_off()
            continue
        sc = ax.scatter(P[:, 0], P[:, 1], c=logf_27, cmap="viridis", s=8, alpha=0.8)
        ax.set_title(title)
    fig_27.colorbar(sc, ax=axes_27[:2].tolist(), label="log10 du nombre d'occurrences (jaune : les plus fréquents)")
    colors_27 = plt.get_cmap("tab10")
    k_27 = 0
    for g, (name, ws) in enumerate(GROUPS_27.items()):
        pts = PCA_GROUPS_27[k_27:k_27 + len(ws)]
        axes_27[2].scatter(pts[:, 0], pts[:, 1], color=colors_27(g), label=name, s=30)
        for (x, y), w in zip(pts, ws):
            axes_27[2].annotate(w, (x, y), fontsize=8, xytext=(3, 3), textcoords="offset points")
        k_27 += len(ws)
    axes_27[2].legend(fontsize=8)
    axes_27[2].set_title("(3) PCA des mots de cinq groupes")
    print(f"Cartes calculées en {fr(time.perf_counter() - start_27, 1)} s.")
    plt.show()

⏳ Ex B2.27 : les cartes attendent ton word2vec entraîné en B2.25 (les vecteurs E_25).


In [62]:
pca_axis_27 = ...         # a) True or False: on map (1), the colour (the frequency) changes steadily along the
                          #    horizontal axis, the frequent words on one side and the rare ones on the other
umap_frequent_27 = ...    # b) True or False: on map (2), the most frequent words (yellow) gather in one region (or a
                          #    few), instead of being spread all over the map

In [63]:
with wb.attempt("B2.27"):
    wb.check("B2.27a", pca_axis_27)
    wb.check("B2.27b", umap_frequent_27)
if globals().get("ok_25") and filled(pca_axis_27, umap_frequent_27) and "PCA_ALL_27" in globals():
    from scipy.stats import spearmanr
    from sklearn.neighbors import NearestNeighbors
    rho_27 = [spearmanr(PCA_ALL_27[:, j], logf_27)[0] for j in (0, 1)]
    print(f"(1) corrélation de rang entre la fréquence et l'axe horizontal : {fr(rho_27[0], 2)} ; l'axe vertical : {fr(rho_27[1], 2)}.")
    if UMAP_ALL_27 is not None:
        _, near_27 = NearestNeighbors(n_neighbors=11).fit(UMAP_ALL_27).kneighbors(UMAP_ALL_27)
        top_27 = np.arange(len(ITOS_25)) < 100
        share_27 = float(top_27[near_27[top_27, 1:]].mean())
        print(f"(2) les 10 voisins sur la carte des 100 mots les plus fréquents sont aussi parmi les 100 plus fréquents "
              f"{fr(100 * share_27, 0)} fois sur 100 (au hasard : {fr(100 * 100 / len(ITOS_25), 0)}).")
    labels_27 = np.array([g for g, ws in GROUPS_27.items() for _ in ws])
    _, near3_27 = NearestNeighbors(n_neighbors=4).fit(PCA_GROUPS_27).kneighbors(PCA_GROUPS_27)
    print("(3) part des 3 voisins sur la carte qui sont du même groupe : "
          + ", ".join(f"{g} {fr(float((labels_27[near3_27[labels_27 == g, 1:]] == g).mean()), 2)}" for g in GROUPS_27))

⏳ Ex B2.27a : pas encore fait (remplace `...` par ta réponse).
⏳ Ex B2.27b : pas encore fait (remplace `...` par ta réponse).


## Partie E · Des embeddings de phrases à la livraison, et le défi

Trois exercices sur ce que font les modèles d'aujourd'hui, puis le défi. Tu calcules des embeddings de phrases avec un modèle pré-entraîné pour chercher des passages de Holmes par le sens (B2.28), tu compares un vecteur statique et un vecteur contextuel du même mot (B2.29), tu livres un tokenizer avec son test de non-régression (B2.30), et tu cherches la meilleure compression de Verne avec 2 000 tokens (B2.31).

B2.28 et B2.29 utilisent `all-MiniLM-L6-v2` (`sentence-transformers/all-MiniLM-L6-v2`, ≈ 90 Mo, Apache 2.0, `data/cards/pretrained.md`), chargé une fois à une révision figée, en `eval()` et sans gradient, par `wbtext.try_sentence_model` : `SM_28.tokenizer` et `SM_28.model` sont son tokenizer et son `AutoModel`. Sans connexion au Hub, ces deux exercices affichent ⏳ « à valider sur Colab ».

### Ex B2.28 — Embeddings de phrases (MiniLM, mean pooling) et recherche sémantique dans Holmes 📦 ★★ ⏱️ 30 min
**Objectif :** calculer des embeddings de phrases par mean pooling avec un modèle pré-entraîné, et chercher des passages par le sens.  
**Prérequis :** Ex B2.23 · B1 (embeddings figés, révision fixée) · fiche §202.10 (« D'une suite de vecteurs à un vecteur de phrase », « Une recherche sémantique en vingt lignes ») · **Fil rouge :** Holmes · **Parcours :** R, C

La cellule suivante charge le modèle et découpe Holmes en 1 055 passages d'au plus 120 mots (`PASSAGES_28`, chacun avec l'indice de son conte dans `STORY_28` et `TITLES_28`), tous sous les 256 tokens que lit MiniLM. Elle calcule une fois l'**index** `INDEX_28` : l'embedding de référence de chaque passage, `(1 055, 384)`, rangé dans le cache `data/downloads/embeddings` (≈ 20 s la première fois, une lecture de fichier ensuite).

**Contrat**
- `embed_28(texts, tok, model, batch_size=32, max_length=256)` : un tableau `float32` `(N, 384)`, un embedding par texte, dans l'ordre :
  - les textes passent par mini-batches de `batch_size` dans `tok(..., padding=True, truncation=True, max_length=max_length, return_tensors="pt")`, puis dans `model`, sans gradient ;
  - le **mean pooling avec le masque d'attention** de la fiche, sur `last_hidden_state` : le padding ne compte pas ;
  - chaque embedding ramené à la **norme 1** ;
  - `max_length=256`, la longueur de la fiche modèle (`tok.model_max_length` dit 512 : piège classique de la fiche).
- `search_28(query, index, k=5)` : `(indices, cosinus)` des `k` lignes de `index` les plus proches de la requête (une `str`), que tu embeddes par `embed_28` avec `SM_28.tokenizer` et `SM_28.model`, et que **ton** `most_similar` (B2.23) cherche.

La cellule de vérification compare `embed_28` à la référence sur quatre textes de longueurs différentes (dont un de plus de 256 tokens), vérifie qu'une phrase courte a le même vecteur seule et au milieu de textes longs, puis cherche six requêtes et affiche, pour chacune, les trois passages trouvés, leur cosinus et leur conte.

Dans tes notes : pour chaque requête, le premier passage est-il du bon conte, et contient-il les mots de la requête ? Essaie une requête à toi, puis la même avec d'autres mots : le classement change-t-il ? Les cosinus vont de 0,4 à 0,65 environ : pourquoi un seuil fixe (« pertinent au-dessus de 0,5 ») serait-il fragile, d'une requête à l'autre et d'un modèle à l'autre ? Pour Verne, il faudrait `multilingual-e5-small` et ses préfixes (`wbtext.try_sentence_model("e5")`, `kind="query"` et `"passage"` : ≈ 470 Mo, à essayer sur Colab).

In [64]:
# all-MiniLM-L6-v2 at its frozen revision (eval(), no gradient), the passages of Holmes and their index
SM_28 = wbtext.try_sentence_model("minilm", "B2.28")
PASSAGES_28, STORY_28 = wbtext.holmes_passages()               # 1 055 passages of at most 120 words, and their story
TITLES_28 = [title for title, _ in wbtext.holmes_stories()]
QUERIES_28 = ["a woman whose fiancé vanished on the morning of the wedding",
              "a deadly snake that comes through a ventilator",
              "men with red hair paid to copy out the encyclopaedia",
              "a precious blue stone found in the crop of a Christmas goose",
              "a beggar in the City who is really a respectable gentleman",
              "a governess asked to cut her hair short"]
INDEX_28 = None
if SM_28 is not None:
    start_28 = time.perf_counter()
    INDEX_28 = wbtext.passage_embeddings(SM_28, PASSAGES_28, "holmes-120")   # (1 055, 384), the reference index
    print(f"{SM_28} ; index de {INDEX_28.shape[0]} passages en {fr(time.perf_counter() - start_28, 1)} s.")

ℹ️ embeddings relus dans le cache (sentence-transformers_all-MiniLM-L6-v2-holmes-120-5582e24b55765ff3.npz)
SentenceModel('minilm': sentence-transformers/all-MiniLM-L6-v2 @ 1110a24, d = 384, max_length = 256) ; index de 1055 passages en 0,0 s.


In [65]:
def embed_28(texts, tok, model, batch_size=32, max_length=256):
    """float32 array (N, 384): the sentence embeddings of the texts, mean pooling with the attention mask, norm 1."""
    raise NotImplementedError("embed_28")


def search_28(query, index, k=5):
    """(indices, cosines): the k rows of `index` (passages embedded, norm 1) closest to the query (a str), embedded by
    embed_28 with SM_28, found by your most_similar (B2.23)."""
    raise NotImplementedError("search_28")

In [66]:
if SM_28 is None:
    print("⏳ Ex B2.28 : la vérification attend all-MiniLM-L6-v2 (connexion au Hub, ou Colab).")
else:
    with checked("B2.28"):
        tok_28, model_28 = SM_28.tokenizer, SM_28.model
        texts_28 = [PASSAGES_28[3], "Holmes smiled.", PASSAGES_28[400], " ".join(PASSAGES_28[10:15])]   # the last: > 256 tokens
        got_28 = embed_28(texts_28, tok_28, model_28)
        if not returned("B2.28", "embed_28", got_28):
            raise StopCheck
        want_28 = SM_28.embed(texts_28)
        same("B2.28", "embed_28 de quatre textes de longueurs différentes, dont un de plus de 256 tokens", got_28, want_28,
             rtol=1e-4, atol=1e-5)
        alone_28 = embed_28(["Holmes smiled."], tok_28, model_28)
        verdict("B2.28", np.allclose(np.asarray(alone_28)[0], np.asarray(got_28)[1], atol=1e-5),
                "une phrase courte a le même vecteur seule ou dans un batch avec des textes longs : le masque retire le padding.",
                "« Holmes smiled. » n'a pas le même vecteur seule et dans un batch : le padding entre-t-il dans ta moyenne ?")
        if not np.allclose(got_28, want_28, rtol=1e-4, atol=1e-5) and np.allclose(np.asarray(got_28)[:3], want_28[:3], atol=1e-5):
            print("💡 seul le texte long diffère : max_length = 256, comme la fiche modèle (pas 512).")
        found_28 = search_28(QUERIES_28[0], INDEX_28, k=5)
        if not returned("B2.28", "search_28", found_28):
            raise StopCheck
        idx_28, sims_28 = pair("B2.28", "search_28", found_28, names=("indices", "cosines"))
        ref_28 = np.argsort(-(INDEX_28 @ SM_28.embed(QUERIES_28[0])[0]), kind="stable")[:5]
        verdict("B2.28", list(np.asarray(idx_28)) == list(ref_28), "search_28 : les 5 passages de la référence, dans l'ordre.",
                f"search_28 : attendu les passages {list(ref_28)}, reçu {list(np.asarray(idx_28))}.")
        for q in QUERIES_28:
            idx, sims = search_28(q, INDEX_28, k=3)
            print(f"\n« {q} »")
            for i, s in zip(idx, sims):
                print(f"   {fr(float(s), 3)}  {TITLES_28[STORY_28[i]]:<28} {PASSAGES_28[i][:90]}…")

⏳ Ex B2.28 : pas encore fait (embed_28).


### Ex B2.29 — Statique contre contextuel : le même mot dans deux phrases 🔬 ★★ ⏱️ 25 min
**Objectif :** mesurer qu'un embedding statique ignore la phrase, alors qu'un embedding contextuel dépend du sens du mot dans sa phrase.  
**Prérequis :** Ex B2.28 · 🧠 Q11 · fiche §202.10 (« Un mot, plusieurs sens ») · **Fil rouge :** Holmes/Verne · **Parcours :** C

Trois mots anglais à deux sens, chacun dans six phrases (`SENTENCES_29`, prises dans Holmes ou écrites dans son style) : « bank » (la banque, la rive), « match » (l'allumette, le mariage), « ring » (la bague, la sonnerie). Pour chaque emploi, tu prends deux vecteurs de MiniLM : la ligne de sa **table d'entrée** (statique, la même dans toutes les phrases) et la sortie de son **dernier bloc** (contextuelle).

**Contrat**
- `contextual_vector_29(sentence, word, tok, model)` : un tableau NumPy `(384,)`, la moyenne des vecteurs de `last_hidden_state` sur les tokens de la **première occurrence** du mot entier `word` dans `sentence` (un mot de plusieurs tokens, comme « Merryweather », en a plusieurs) ; trouve-les avec les positions que rend `tok(sentence, return_offsets_mapping=True)`, sans les tokens spéciaux ; sans gradient.
- `static_vector_29(word, tok, model)` : `(384,)`, la moyenne des lignes de `model.get_input_embeddings().weight` pour les tokens de `word` (sans tokens spéciaux).
- `sense_means_29(vectors, senses)` : le couple `(moyenne des cosinus des paires d'emplois de même sens, moyenne des cosinus des paires de sens différents)`, deux `float` ; chaque paire d'emplois compte une fois.

La première cellule de vérification compare tes vecteurs à ceux de référence (`SM_29.word_vector`, `SM_29.static_vector`) et `sense_means_29` à un petit calcul visible. La suivante calcule, pour chaque mot, les cosinus des six emplois (une carte de chaleur par mot) et le tableau des moyennes, statiques et contextuelles.

Dans tes notes : que valent les cosinus statiques, et pourquoi n'est-ce pas une mesure ? Pour chaque mot, l'écart entre « même sens » et « sens différents » : lequel des trois sépare le mieux ses deux sens, lequel le moins ? Pourquoi tous les cosinus contextuels sont-ils élevés (au-dessus de 0,6), même entre deux sens ? Que te dit ce chapitre de l'entraînement de MiniLM (pour des phrases entières, §202.10) ?

In [67]:
# Three words with two senses each, six sentences each (taken from Holmes or written in its style)
SENTENCES_29 = {
    "bank": [("argent", "This fellow Merryweather is a bank director, and personally interested in the matter."),
             ("argent", "I was seated in my office at the bank when a card was brought in to me."),
             ("argent", "He had some fourteen thousand pounds, which lay to his credit at the bank."),
             ("rive", "As of the swimmer who leaves the bank, she hurried across the road."),
             ("rive", "He was lying on the grassy bank of the pool, close to the water."),
             ("rive", "The boat was tied to a tree on the bank of the river.")],
    "match": [("allumette", "Holmes sprang from the bed, struck a match, and lashed at the bell-pull."),
              ("allumette", "In her right hand was found the charred stump of a match."),
              ("allumette", "By the light of her match she caught a hurried glimpse of him."),
              ("mariage", "My stepfather has offered no opposition to the match, and we are to be married."),
              ("mariage", "Her father thought it a very good match for her, and gave his consent."),
              ("mariage", "It was a love match, and the wedding was fixed for the spring.")],
    "ring": [("bijou", "He slipped an emerald snake ring from his finger and held it out."),
             ("bijou", "She put a new wedding ring upon the top of the pile."),
             ("bijou", "He wore a heavy gold ring with a large stone on his little finger."),
             ("sonnerie", "There came a ring to my bell, about the hour when a man gives his first yawn."),
             ("sonnerie", "I think that I heard a ring at the door, said Holmes."),
             ("sonnerie", "There was a sharp ring at the bell, and the maid went to open.")]}
SM_29 = SM_28 if "SM_28" in globals() else wbtext.try_sentence_model("minilm", "B2.29")

In [68]:
def contextual_vector_29(sentence, word, tok, model):
    """The output of the last block (last_hidden_state) for `word` in `sentence`: the mean of the vectors of the tokens
    of its first occurrence (found with the offsets of the tokenizer), a NumPy array (384,)."""
    raise NotImplementedError("contextual_vector_29")


def static_vector_29(word, tok, model):
    """The row of the input table (model.get_input_embeddings()) of `word`, averaged over its tokens: (384,)."""
    raise NotImplementedError("static_vector_29")


def sense_means_29(vectors, senses):
    """(mean cosine of the pairs of uses of the SAME sense, mean cosine of the pairs of DIFFERENT senses), two floats."""
    raise NotImplementedError("sense_means_29")

In [69]:
if SM_29 is None:
    print("⏳ Ex B2.29 : l'expérience attend all-MiniLM-L6-v2 (connexion au Hub, ou Colab).")
else:
    with checked("B2.29"):
        tok_29, model_29 = SM_29.tokenizer, SM_29.model
        probe_29 = "The bank-note was found on the bank, near the river."      # two « bank »: the first one counts
        v_29 = contextual_vector_29(probe_29, "bank", tok_29, model_29)
        if not returned("B2.29", "contextual_vector_29", v_29):
            raise StopCheck
        same("B2.29", "contextual_vector_29 de « bank » (la première occurrence)", v_29, SM_29.word_vector(probe_29, "bank"),
             rtol=1e-4, atol=1e-5)
        same("B2.29", "contextual_vector_29 d'un mot de plusieurs tokens (« Merryweather »)",
             contextual_vector_29(SENTENCES_29["bank"][0][1], "Merryweather", tok_29, model_29),
             SM_29.word_vector(SENTENCES_29["bank"][0][1], "Merryweather"), rtol=1e-4, atol=1e-5)
        same("B2.29", "static_vector_29 de « bank » et de « Merryweather »",
             np.stack([static_vector_29(w, tok_29, model_29) for w in ("bank", "Merryweather")]),
             np.stack([SM_29.static_vector(w) for w in ("bank", "Merryweather")]), rtol=1e-5, atol=1e-6)
        means_29 = sense_means_29(np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [-1.0, 0.0]]), ["a", "a", "b", "b"])
        if returned("B2.29", "sense_means_29", means_29):
            want_29 = (np.mean([math.sqrt(0.5), 0.0]), np.mean([math.sqrt(0.5), -math.sqrt(0.5), 0.0, -1.0]))
            same("B2.29", "sense_means_29 sur quatre vecteurs du plan (sens a, a, b, b)", means_29, want_29)

if SM_29 is not None and written(lambda: contextual_vector_29("a bank", "bank", SM_29.tokenizer, SM_29.model)) \
        and written(lambda: sense_means_29(np.eye(3), ["a", "a", "b"])):
    with checked("B2.29"):
        rows_29 = []
        fig_29, axes_29 = plt.subplots(1, 3, figsize=(15, 4.4))
        for ax, (word, uses) in zip(axes_29, SENTENCES_29.items()):
            senses = [s for s, _ in uses]
            ctx = np.stack([contextual_vector_29(s, word, SM_29.tokenizer, SM_29.model) for _, s in uses])
            stat = np.stack([static_vector_29(word, SM_29.tokenizer, SM_29.model) for _ in uses])
            same_ctx, cross_ctx = sense_means_29(ctx, senses)
            same_st, cross_st = sense_means_29(stat, senses)
            rows_29.append({"mot": word, "contextuel, même sens": same_ctx, "contextuel, sens différents": cross_ctx,
                            "statique, même sens": same_st, "statique, sens différents": cross_st})
            U = ctx / np.linalg.norm(ctx, axis=1, keepdims=True)
            im = ax.imshow(U @ U.T, vmin=0.6, vmax=1.0, cmap="viridis")
            ax.set_xticks(range(6), [f"{s[:4]}{k % 3 + 1}" for k, s in enumerate(senses)], fontsize=8)
            ax.set_yticks(range(6), [f"{s[:4]}{k % 3 + 1}" for k, s in enumerate(senses)], fontsize=8)
            ax.set_title(f"« {word} »")
            ax.grid(False)
        fig_29.colorbar(im, ax=axes_29.tolist(), label="cosinus des vecteurs contextuels")
        table_29 = pd.DataFrame(rows_29).set_index("mot")
        print(table_29.to_string(float_format=lambda x: f"{x:.3f}"))
        verdict("B2.29", np.allclose(table_29[["statique, même sens", "statique, sens différents"]], 1.0),
                "le vecteur statique d'un mot est le même dans toutes les phrases : cosinus 1, quel que soit le sens.",
                "le vecteur statique doit être la même ligne de la table dans toutes les phrases (cosinus 1).")
        plt.show()

⏳ Ex B2.29 : pas encore fait (contextual_vector_29).


### Ex B2.30 — Sauvegarder un tokenizer en JSON et écrire son test de non-régression 🛠️ ★★ ⏱️ 20 min
**Objectif :** livrer un tokenizer comme un artefact versionné, et le protéger par un test de non-régression qui échoue dès qu'il change.  
**Prérequis :** Ex B2.17 · fiche §202.4, §202.5 (« Les tokenizers des modèles, lus dans leur `tokenizer.json` ») · ch. 0A (tests pytest) · **Fil rouge :** Holmes/Verne · **Parcours :** R, C

Un modèle est entraîné avec **un** tokenizer : si ses fusions changent, ses identifiants changent, et le modèle lit n'importe quoi sans le moindre message. En entreprise, le tokenizer est donc livré comme un fichier, à côté des poids, et un **test de non-régression** vérifie qu'il redonne toujours les mêmes identifiants sur un texte de référence figé.

La cellule suivante prépare `TOK_30`, un `BPETokenizer` octet de **ta** librairie, 1 000 tokens appris sur `VERNE_TRAIN` (par `fast_byte_bpe`), le texte de référence `REFERENCE_TEXT_30` (un passage de la validation de Verne, puis « € », un emoji et des idéogrammes), et le dossier de livraison `DELIVERY_30` (à côté de ton notebook dans `mon_travail`, sinon un dossier temporaire).

**Contrat**
- `deliver_30(tok, folder, reference_text)` : écrit `folder/tokenizer.json` (par `tok.save`, fourni) et `folder/golden.json`, un dictionnaire JSON qui contient au moins `"text"` (le texte de référence) et `"ids"` (ses identifiants figés), et ce que ton test veut vérifier d'autre ; rend le couple de leurs chemins `(tokenizer.json, golden.json)`.
- `test_tokenizer_regression_30()` : un test pytest qui recharge le tokenizer par `BPETokenizer.load(TOKENIZER_PATH)`, puis vérifie avec des `assert` que l'encodage du texte de `GOLDEN_PATH` redonne les ids figés, que leur décodage redonne le texte, et que le nombre de tokens n'a pas changé. `BPETokenizer`, `TOKENIZER_PATH`, `GOLDEN_PATH`, `json` et `Path` sont fournis au test, qui tourne dans un fichier à part (`wb.run_pytest`) : il ne doit pas lire d'autre variable du notebook.

La cellule de vérification livre `TOK_30`, relit les deux fichiers, puis lance ton test sur le tokenizer livré (il doit passer) et sur trois copies abîmées de `tokenizer.json` (il doit échouer sur chacune).

Dans tes notes : quels fichiers committerais-tu à côté des poids d'un modèle, et pourquoi le texte de référence contient-il des accents, un emoji et des idéogrammes ? Une des trois copies abîmées garde les mêmes ids pour la plupart des textes anglais : laquelle, et pourquoi ton test la voit-il quand même ? Quand faut-il **régénérer** `golden.json`, et qui doit le décider ?

In [70]:
# The tokenizer to deliver: a byte-level BPE of 1 000 tokens of your library, learned on VERNE_TRAIN, and the frozen
# reference text of its non-regression test (French, emoji, ideographs, a « € »)
import tempfile
from pathlib import Path

REFERENCE_TEXT_30 = VERNE_VAL[:1500] + "\n« Ça coûte 15 € », dit-il 🙂 漢字.\n"
_here_30 = Path.cwd().resolve()
DELIVERY_30 = (_here_30 / "livraison_b2_30") if "mon_travail" in _here_30.parts else Path(tempfile.mkdtemp(prefix="wb_b2_30_"))
TOK_30 = fast_byte_bpe(VERNE_TRAIN, 1000) if bpe_written() else None
_package_30 = Path(mylearn.tokenizer.__file__).resolve().parent    # mon_travail/mylearn (or the reference)
LIBRARY_30 = (str(_package_30.parent), f"{_package_30.name}.tokenizer")
print(f"Dossier de livraison : {DELIVERY_30}")


def broken_copies_30(folder):
    """Three broken copies of the delivered tokenizer.json (for the check): two merges swapped, the last merge removed,
    the pre-tokenizer changed. Returns {description: path}."""
    state = json.loads((Path(folder) / "tokenizer.json").read_text(encoding="utf-8"))
    out = {}
    for name, change in (("deux fusions échangées", lambda s: s["merges"].__setitem__(slice(10, 12), s["merges"][10:12][::-1])),
                         ("la dernière fusion retirée", lambda s: s["merges"].pop()),
                         ("le pré-tokenizer 'whitespace' au lieu de 'gpt2'", lambda s: s["params"].__setitem__("pre_tokenizer", "whitespace"))):
        broken = json.loads(json.dumps(state))
        change(broken)
        target = Path(tempfile.mkdtemp(prefix="wb_b2_30_broken_")) / "tokenizer.json"
        target.write_text(json.dumps(broken), encoding="utf-8")
        out[name] = target
    return out

MylearnMissingError: ta librairie mylearn n'existe pas encore dans mon_travail/mylearn/ ; lance d'abord : python tools/start_chapter.py B2

In [ ]:
def deliver_30(tok, folder, reference_text):
    """Write folder/tokenizer.json (tok.save) and folder/golden.json ({"text": reference_text, "ids": its ids}); return
    (path of tokenizer.json, path of golden.json)."""
    raise NotImplementedError("deliver_30")


def test_tokenizer_regression_30():
    """The non-regression test: BPETokenizer.load(TOKENIZER_PATH) must give the frozen ids of GOLDEN_PATH, decode them back
    to the text, and keep its number of tokens."""
    raise NotImplementedError("test_tokenizer_regression_30")

In [ ]:
import inspect

if TOK_30 is None:
    print("⏳ Ex B2.30 : la livraison attend ton BPETokenizer (B2.16 et B2.17).")
else:
    with checked("B2.30"):
        paths_30 = deliver_30(TOK_30, DELIVERY_30, REFERENCE_TEXT_30)
        if not returned("B2.30", "deliver_30", paths_30):
            raise StopCheck
        tok_path_30, golden_path_30 = (Path(p) for p in pair("B2.30", "deliver_30", paths_30, names=("tokenizer.json", "golden.json")))
        reloaded_30 = type(TOK_30).load(tok_path_30)
        golden_30 = json.loads(golden_path_30.read_text(encoding="utf-8"))
        verdict("B2.30", reloaded_30.merges_ == TOK_30.merges_ and golden_30.get("text") == REFERENCE_TEXT_30
                and golden_30.get("ids") == TOK_30.encode(REFERENCE_TEXT_30),
                f"livraison : {tok_path_30.name} se recharge à l'identique, {golden_path_30.name} contient le texte et ses "
                f"{len(golden_30.get('ids', []))} ids figés.",
                "tokenizer.json doit se recharger avec BPETokenizer.load, et golden.json contenir le texte de référence "
                "(clé 'text') et ses ids (clé 'ids').")
    if "NotImplementedError" in inspect.getsource(test_tokenizer_regression_30):
        print("⏳ Ex B2.30 : écris ton test de non-régression.")
    elif "tok_path_30" in globals():
        def preamble_30(tok_path):
            return (f"import json\nimport sys\nfrom pathlib import Path\nsys.path.insert(0, {LIBRARY_30[0]!r})\n"
                    f"from {LIBRARY_30[1]} import BPETokenizer\nTOKENIZER_PATH = {str(tok_path)!r}\n"
                    f"GOLDEN_PATH = {str(golden_path_30)!r}")
        print("Ton test, sur le tokenizer livré (il doit passer) :")
        good_30 = wb.run_pytest([test_tokenizer_regression_30], preamble=preamble_30(tok_path_30))
        verdict("B2.30", good_30.ok, "ton test passe sur le tokenizer livré.", "ton test doit passer sur le tokenizer livré "
                "(la sortie de pytest est au-dessus).")
        for why_30, path_30 in (broken_copies_30(DELIVERY_30).items() if good_30.ok else ()):
            caught_30 = not wb.run_pytest([test_tokenizer_regression_30], preamble=preamble_30(path_30), quiet=True).ok
            verdict("B2.30", caught_30, f"ton test attrape un tokenizer.json abîmé : {why_30}.",
                    f"ton test laisse passer un tokenizer.json abîmé : {why_30}.")

### Ex B2.31 — Défi : la meilleure compression sans perte de Verne avec 2 000 tokens, en moins de 2 min de CPU 🏆 ★★★★ ⏱️ 90 min
**Objectif :** trouver, avec ce que le chapitre a montré, un tokenizer de 2 000 tokens qui compresse Verne nettement mieux que le BPE de B2.20.  
**Prérequis :** Ex B2.20, B2.21 · fiche §202.3 à §202.6 · tout le chapitre · **Fil rouge :** Verne · **Parcours :** C

En B2.20, un BPE octet de 2 000 tokens avec le pré-tokenizer `'gpt2'`, appliqué par rang, a fait **3,001 octets par token** sur `VERNE_VAL`. Le défi : **au moins 3,20**, toujours **sans perte**, avec **2 000 tokens au plus** et en **moins de 2 minutes** de CPU.

Écris `method_31(text)` : elle reçoit `VERNE_TRAIN` et rend un tokenizer, un objet qui a :
- `vocab_` : un dictionnaire `{id: bytes}` d'au plus 2 000 entrées, tokens spéciaux compris (celui de ton `BPETokenizer` convient) ;
- `encode(text)` : la liste des ids du texte, tous des clés de `vocab_` ;
- `decode(ids)` : le texte.

Tout est permis : ton `BPETokenizer`, la bibliothèque `tokenizers`, d'autres pré-tokenizers, une autre façon d'appliquer un vocabulaire, une idée à toi. `evaluate_31(method_31)` (cellule suivante) vérifie quatre règles :
1. **2 000 tokens au plus** dans `vocab_` ;
2. **sans perte** : les octets des tokens, mis bout à bout, redonnent exactement la validation, et `decode` aussi : un id vaut toujours les mêmes octets, quel que soit son voisinage ;
3. **rien de la validation** : ta méthode n'apprend que sur le texte qu'elle reçoit. Pendant la course, toutes les variables globales qui contiennent la validation (`VERNE_VAL` et celles des exercices précédents) valent un leurre, et un token de 4 octets ou plus qui serait dans la validation mais nulle part dans `VERNE_TRAIN` est signalé ;
4. **moins de 2 minutes** pour l'entraînement et l'encodage de la validation, mesurés ensemble.

Puis elle affiche ton score : les octets UTF-8 de la validation divisés par son nombre de tokens. Règle d'honneur : la validation sert à mesurer, jamais à apprendre.

Dans tes notes : tes idées, leur gain sur la validation et leur temps ; ce que coûterait ta méthode à un modèle de langue (la longueur des séquences, la table d'embedding, les tokens rares, B2.20) ; et pourquoi un tokenizer qui compresse mieux n'est pas toujours un meilleur tokenizer.

In [ ]:
# The challenge: the best lossless compression of VERNE_VAL with at most 2 000 tokens, in less than 2 minutes of CPU
GOAL_31, LIMIT_31, MAX_TOKENS_31 = 3.20, 120.0, 2000     # bytes per token to reach; seconds; tokens (special included)
BASELINE_31 = 3.001        # B2.20: byte-level BPE 'gpt2' of 2 000 tokens, by rank, learned on VERNE_TRAIN, on VERNE_VAL
BASELINE_CONTROL_31 = 2.955   # the same BPE learned on the first 8/9 of VERNE_TRAIN, on its last ninth (the control)


def _make_evaluate_31(validation, train):
    """evaluate_31, with the validation text and the control text kept here only (the global variables that hold them
    are replaced by a decoy while your method runs)."""
    cut = wbtext.split_point(train, 8 / 9)
    texts = {"validation": (validation, train), "contrôle": (train[cut:], train[:cut])}   # name: (measured, learned on)
    decoy = " ".join(sorted(set(train.split()))[:6000])       # words of the training text, in alphabetical order

    def marker(text):
        at = next(i for i in range(1000, len(text)) if not set(text[i:i + 60]) & set("\n'\"\\"))
        return text[at:at + 60]                                 # a piece of the text, the same in a repr

    def holds(value, markers):
        try:
            shown = value if isinstance(value, str) else str(value)[:5_000_000]
        except Exception:  # noqa: BLE001 - an object without a readable str
            return False
        return any(m in shown for m in markers)

    def run(method, name):
        """Train the method on the text that goes with `name` and encode the text measured, the global variables that
        hold a measured text being replaced by the decoy. Returns (tokenizer, ids, seconds), or None."""
        g = globals()
        measured, learned = texts[name]
        markers = [marker(validation)] + ([marker(measured)] if name == "contrôle" else [])
        hidden = {n: v for n, v in list(g.items()) if not n.startswith("_") and isinstance(v, (str, dict, list, tuple))
                  and holds(v, markers)}
        try:
            for n in hidden:
                g[n] = decoy
            start = time.perf_counter()
            tok = method(learned)
            fit_seconds = time.perf_counter() - start
            vocab = getattr(tok, "vocab_", None)
            if not isinstance(vocab, dict) or not vocab or not all(isinstance(b, bytes) for b in vocab.values()):
                print("❌ Ex B2.31 : le tokenizer rendu doit avoir un vocab_, un dict {id: bytes} (comme ton BPETokenizer).")
                return None
            start = time.perf_counter()
            ids = tok.encode(measured)
            return tok, ids, fit_seconds + time.perf_counter() - start
        finally:
            for n, v in hidden.items():
                g[n] = v

    def evaluate_31(method):
        """method(text) -> tokenizer: the four rules of the statement, then the bytes per token on the validation."""
        out = run(method, "validation")
        if out is None:
            return None
        tok, ids, seconds = out
        vocab, val_bytes = tok.vocab_, validation.encode("utf-8")
        if not isinstance(ids, list) or not ids or not all(isinstance(i, (int, np.integer)) and i in vocab for i in ids):
            print("❌ Ex B2.31 : encode doit rendre une liste d'ids entiers, tous des clés de vocab_.")
            return None
        lossless = b"".join(vocab[i] for i in ids) == val_bytes
        decoded = tok.decode(ids) == validation if hasattr(tok, "decode") else False
        score = len(val_bytes) / len(ids)
        print(f"{len(vocab)} tokens ; validation : {len(ids):_} tokens pour {len(val_bytes):_} octets, soit {fr(score, 3)} "
              f"octets par token ; entraînement + encodage : {fr(seconds, 1)} s.".replace("_", " "))
        verdict("B2.31", len(vocab) <= MAX_TOKENS_31, f"{len(vocab)} tokens, au plus {MAX_TOKENS_31}.",
                f"{len(vocab)} tokens dans vocab_ : le défi en permet {MAX_TOKENS_31}, tokens spéciaux compris.")
        verdict("B2.31", lossless and decoded, "sans perte : les octets des tokens redonnent la validation, et decode aussi.",
                "pas sans perte : " + ("les octets des tokens, mis bout à bout, ne redonnent pas la validation" if not lossless
                                       else "decode(ids) ne redonne pas la validation") + ".")
        verdict("B2.31", seconds <= LIMIT_31, f"en {fr(seconds, 1)} s, sous les {LIMIT_31:.0f} s.",
                f"{fr(seconds, 1)} s : le défi en permet {LIMIT_31:.0f} (entraînement + encodage de la validation).")
        # the control: the same method, learned without the last ninth of VERNE_TRAIN, measured on it
        honest, control = False, run(method, "contrôle")
        if control is not None:
            measured, (c_tok, c_ids, _) = texts["contrôle"][0], control
            if isinstance(c_ids, list) and c_ids and b"".join(c_tok.vocab_.get(i, b"?") for i in c_ids) == measured.encode("utf-8"):
                gain, c_gain = score / BASELINE_31, len(measured.encode("utf-8")) / len(c_ids) / BASELINE_CONTROL_31
                honest = gain - c_gain <= 0.02
                verdict("B2.31", honest, f"contrôle : sur un texte tenu à l'écart, ta méthode gagne autant sur la référence "
                        f"de B2.20 ({fr(100 * (c_gain - 1), 1)} %) que sur la validation ({fr(100 * (gain - 1), 1)} %).",
                        f"contrôle : ta méthode gagne {fr(100 * (gain - 1), 1)} % sur la validation, mais seulement "
                        f"{fr(100 * (c_gain - 1), 1)} % sur un texte tenu à l'écart : apprend-elle autre chose que le texte "
                        "qu'elle reçoit (la validation, une variable globale) ?")
            else:
                print("❌ Ex B2.31 : contrôle : sur le texte tenu à l'écart, ta méthode n'est pas sans perte.")
        ok = len(vocab) <= MAX_TOKENS_31 and lossless and decoded and seconds <= LIMIT_31 and honest
        verdict("B2.31", ok and score >= GOAL_31, f"défi réussi : {fr(score, 3)} octets par token (objectif {fr(GOAL_31, 2)}, "
                f"référence de B2.20 {fr(BASELINE_31, 3)}).",
                f"pas encore : {fr(score, 3)} octets par token, il en faut {fr(GOAL_31, 2)} en respectant les quatre règles.")
        return {"bytes_per_token": score, "seconds": seconds, "tokens": len(vocab), "honest": honest,
                "ok": ok and score >= GOAL_31}

    return evaluate_31


evaluate_31 = _make_evaluate_31(VERNE_VAL, VERNE_TRAIN)

In [ ]:
def method_31(text):
    """Your method: learn on `text` (VERNE_TRAIN) ONLY a tokenizer with vocab_ ({id: bytes}, at most 2 000 entries),
    encode(text) -> list of ids and decode(ids) -> str, lossless."""
    raise NotImplementedError("method_31")

In [ ]:
with checked("B2.31"):
    need("B2.31", "method_31")
    result_31 = evaluate_31(method_31)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu compter les octets et les points de code d'un texte, et écrire un caractère en UTF-8 ?
2. Sais-tu dérouler un entraînement BPE à la main, départage compris, puis encoder un mot nouveau par rang ?
3. Sais-tu dire quelles lignes d'une table d'embedding reçoivent un gradient, et calculer une PPMI ?
4. Sais-tu chiffrer ce que coûtent une table d'embedding et un texte selon sa langue ?

**Pour aller plus loin** : le chapitre 6 du cours LLM de Hugging Face et `minbpe` d'A. Karpathy, cités dans la fiche. La suite : le bonus B3, attention et Transformers.